# Global Wheat Detection — Data Exploration, Cleansing, Formatting & Augmentation

**Competition:** https://www.kaggle.com/competitions/global-wheat-detection/overview

**Internship timeline**
- Week 1 (this notebook): data exploration, cleansing, formatting, train/val split, and augmentation pipeline
- Week 3 & 4 (next notebook): fine-tuning and optimization, using the split + augmentation pipeline built here

This notebook covers:
1. Necessary libraries and data paths
2. Load and show data
3. Images with/without bounding boxes + example of each + total box count
4. Applying bounding boxes to sample images
5. Number of images by bounding-box count
6. Images and boxes per source
7. Bounding box area distribution + outlier detection (small / negative / large), calibrated visually
8. Aspect ratio distribution
9. Extracting and separating bounding box attributes (incl. duplicate/interchangeable box detection via IoU, and resolving which duplicate copy to drop)
10. Train / validation split (source-stratified, at image level)
11. Data augmentation pipeline (Albumentations) with visual sanity checks
12. Conversion to YOLO-format dataset (split-aware; outliers and duplicate boxes excluded)
13. YOLO label sanity check


## 1. Necessary Libraries and Data Paths

In [ ]:
import os
print(os.listdir("/kaggle/input/competitions"))

In [ ]:
import os
import ast
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image

import albumentations as A
from sklearn.model_selection import train_test_split

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

CLASS_NAME = "wheat"
CLASS_ID = 0

# ----- Data paths -----
# Adjust DATA_DIR to wherever the competition data was downloaded/extracted.
# Expected structure:
#   DATA_DIR/train.csv
#   DATA_DIR/train/*.jpg
#   DATA_DIR/test/*.jpg
DATA_DIR = Path("/kaggle/input/competitions/global-wheat-detection")          # e.g. Path("/kaggle/input/global-wheat-detection")
TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_IMG_DIR = DATA_DIR / "train"
TEST_IMG_DIR = DATA_DIR / "test"

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
YOLO_DIR = OUTPUT_DIR / "yolo_dataset"

assert TRAIN_CSV.exists(), f"train.csv not found at {TRAIN_CSV.resolve()} - update DATA_DIR"
print("Using data directory:", DATA_DIR.resolve())

## 2. Load and Show Data

In [ ]:
df_raw = pd.read_csv(TRAIN_CSV)
print("train.csv shape:", df_raw.shape)
display(df_raw.head())
df_raw.info()
display(df_raw.describe(include="all"))

In [ ]:
all_train_images = sorted(p.stem for p in TRAIN_IMG_DIR.glob("*.jpg"))
print(f"Total images in train folder      : {len(all_train_images)}")
print(f"Total rows (boxes) in train.csv   : {len(df_raw)}")
print(f"Unique image_ids in train.csv     : {df_raw['image_id'].nunique()}")
print(f"Unique sources                    : {df_raw['source'].unique().tolist()}")

In [ ]:
# Peek at a handful of raw images (no boxes yet)
sample_ids = random.sample(all_train_images, 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_id in zip(axes.ravel(), sample_ids):
    img = Image.open(TRAIN_IMG_DIR / f"{img_id}.jpg")
    ax.imshow(img)
    ax.set_title(img_id, fontsize=9)
    ax.axis("off")
plt.suptitle("Sample raw training images")
plt.tight_layout()
plt.show()

## 3. Images With / Without Bounding Boxes

In [ ]:
def parse_bbox(bbox_str):
    """Parse the string-encoded bbox '[xmin, ymin, w, h]' into floats."""
    return ast.literal_eval(bbox_str)

df = df_raw.copy()
bbox_arr = np.array(df["bbox"].apply(parse_bbox).tolist())
df["x_min"] = bbox_arr[:, 0]
df["y_min"] = bbox_arr[:, 1]
df["box_width"] = bbox_arr[:, 2]
df["box_height"] = bbox_arr[:, 3]

images_with_boxes = set(df["image_id"].unique())
images_without_boxes = sorted(set(all_train_images) - images_with_boxes)

print(f"Images WITH at least one bounding box : {len(images_with_boxes)}")
print(f"Images WITHOUT any bounding box        : {len(images_without_boxes)}")
print(f"Total images                           : {len(all_train_images)}")
print(f"Total bounding boxes                   : {len(df)}")

In [ ]:
example_with = df["image_id"].iloc[0]
example_without = images_without_boxes[0] if images_without_boxes else None

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

img = np.array(Image.open(TRAIN_IMG_DIR / f"{example_with}.jpg"))
axes[0].imshow(img)
for _, row in df[df["image_id"] == example_with].iterrows():
    rect = patches.Rectangle((row.x_min, row.y_min), row.box_width, row.box_height,
                              linewidth=2, edgecolor="red", facecolor="none")
    axes[0].add_patch(rect)
axes[0].set_title(f"WITH boxes: {example_with}")
axes[0].axis("off")

if example_without:
    img2 = np.array(Image.open(TRAIN_IMG_DIR / f"{example_without}.jpg"))
    axes[1].imshow(img2)
    axes[1].set_title(f"WITHOUT boxes: {example_without}")
else:
    axes[1].text(0.5, 0.5, "No image without boxes found", ha="center")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 4. Applying Bounding Boxes to Sample Images

In [ ]:
def draw_boxes(image_id, boxes_df, ax=None, color="red", linewidth=2):
    img = np.array(Image.open(TRAIN_IMG_DIR / f"{image_id}.jpg"))
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img)
    boxes = boxes_df[boxes_df["image_id"] == image_id]
    for _, row in boxes.iterrows():
        rect = patches.Rectangle((row.x_min, row.y_min), row.box_width, row.box_height,
                                  linewidth=linewidth, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
    ax.set_title(f"{image_id}  ({len(boxes)} boxes)", fontsize=10)
    ax.axis("off")
    return ax

sample_with_boxes = random.sample(list(images_with_boxes), 6)
fig, axes = plt.subplots(2, 3, figsize=(16, 11))
for ax, img_id in zip(axes.ravel(), sample_with_boxes):
    draw_boxes(img_id, df, ax=ax)
plt.suptitle("Sample images with bounding boxes")
plt.tight_layout()
plt.show()

## 5. Number of Images by Bounding-Box Count

In [ ]:
box_counts_per_image = df.groupby("image_id").size()
# include images that have zero boxes
box_counts_full = box_counts_per_image.reindex(all_train_images, fill_value=0)

plt.figure(figsize=(12, 6))
sns.histplot(box_counts_full, bins=range(0, int(box_counts_full.max()) + 2), color="steelblue")
plt.xlabel("Number of bounding boxes per image")
plt.ylabel("Number of images")
plt.title("Distribution of bounding-box counts per image")
plt.show()

print(box_counts_full.describe())
print(f"Images with 0 boxes: {(box_counts_full == 0).sum()}")

## 6. Images and Boxes per Source

In [ ]:
boxes_per_source = df["source"].value_counts()
images_per_source = df.groupby("source")["image_id"].nunique().reindex(boxes_per_source.index)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x=images_per_source.index, y=images_per_source.values,
            hue=images_per_source.index, palette="viridis", legend=False, ax=axes[0])
axes[0].set_title("Number of images per source")
axes[0].set_ylabel("Images")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(x=boxes_per_source.index, y=boxes_per_source.values,
            hue=boxes_per_source.index, palette="magma", legend=False, ax=axes[1])
axes[1].set_title("Number of bounding boxes per source")
axes[1].set_ylabel("Bounding boxes")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

display(pd.DataFrame({"images": images_per_source, "boxes": boxes_per_source}))

In [ ]:
# A few example images (with boxes drawn) from each source
sources = boxes_per_source.index.tolist()
n_examples = 3

fig, axes = plt.subplots(len(sources), n_examples, figsize=(5 * n_examples, 5 * len(sources)))
if len(sources) == 1:
    axes = axes[None, :]

for row_i, source in enumerate(sources):
    source_ids = df.loc[df["source"] == source, "image_id"].unique()
    sample_ids = random.sample(list(source_ids), min(n_examples, len(source_ids)))

    for col_i in range(n_examples):
        ax = axes[row_i, col_i]
        if col_i >= len(sample_ids):
            ax.axis("off")
            continue
        img_id = sample_ids[col_i]
        draw_boxes(img_id, df, ax=ax, color="lime")
        ax.set_title(f"{source} | {img_id}", fontsize=9)

plt.suptitle("Sample images with bounding boxes, per source", fontsize=16, y=1.001)
plt.tight_layout()
plt.show()

## 7. Bounding Box Area Distribution & Outliers

Box area is compared to the size of its **own** image (`area_ratio`), not a fixed
pixel count — a "big" box on a small image and a "big" box on a large image mean
very different things. Thresholds below were picked by visually calibrating
against real examples (see 7.2), not guessed upfront.

### 7.1 Compute Box Area Relative to Image Size

In [ ]:
df["area"] = df["box_width"] * df["box_height"]
df["image_area"] = df["width"] * df["height"]
df["area_ratio"] = df["area"] / df["image_area"]

NEG_DIM_MASK = (df["box_width"] <= 0) | (df["box_height"] <= 0)
print(f"Negative/zero-dimension boxes: {NEG_DIM_MASK.sum()}")

### 7.2 Area-Ratio Bucket Calibration

Before committing to small/large thresholds, look at real examples across the
full range of `area_ratio` to see where "normal" actually stops.

In [ ]:
df_valid = df[~NEG_DIM_MASK].copy()
df_valid["area_pct"] = df_valid["area_ratio"] * 100

bucket_edges = [0, 1, 10, 20, 30, 40, 50, 100]
bucket_labels = ["<1%", "1-10%", "10-20%", "20-30%", "30-40%", "40-50%", "50%+"]
df_valid["area_bucket"] = pd.cut(df_valid["area_pct"], bins=bucket_edges,
                                  labels=bucket_labels, right=False)

bucket_counts = df_valid["area_bucket"].value_counts().reindex(bucket_labels)
bucket_pct = (bucket_counts / len(df_valid) * 100).round(3)
display(pd.DataFrame({"count": bucket_counts, "% of all boxes": bucket_pct}))

plt.figure(figsize=(10, 5))
sns.barplot(x=bucket_counts.index, y=bucket_counts.values,
            hue=bucket_counts.index, palette="crest", legend=False)
plt.yscale("log")
plt.ylabel("Number of boxes (log scale)")
plt.xlabel("Box area as % of image area")
plt.title("Box count per area-ratio bucket")
plt.show()

In [ ]:
def show_bucket_examples(df_valid, bucket_labels, n_per_bucket=3, seed=RANDOM_SEED):
    n_rows = len(bucket_labels)
    fig, axes = plt.subplots(n_rows, n_per_bucket,
                              figsize=(5 * n_per_bucket, 5 * n_rows))
    if n_rows == 1:
        axes = axes[None, :]

    for row_i, bucket in enumerate(bucket_labels):
        bucket_df = df_valid[df_valid["area_bucket"] == bucket]
        sampled_rows = bucket_df.sample(min(n_per_bucket, len(bucket_df)), random_state=seed) \
            if len(bucket_df) > 0 else bucket_df

        for col_i in range(n_per_bucket):
            ax = axes[row_i, col_i]
            ax.axis("off")
            if col_i >= len(sampled_rows):
                if col_i == 0 and len(sampled_rows) == 0:
                    ax.text(0.5, 0.5, f"No boxes in {bucket}", ha="center", va="center")
                continue

            row = sampled_rows.iloc[col_i]
            img = np.array(Image.open(TRAIN_IMG_DIR / f"{row.image_id}.jpg"))
            ax.imshow(img)
            rect = patches.Rectangle((row.x_min, row.y_min), row.box_width, row.box_height,
                                      linewidth=3, edgecolor="lime", facecolor="none")
            ax.add_patch(rect)
            ax.set_title(f"{bucket} | {row.image_id}\n{row.area_pct:.2f}% of image", fontsize=9)

    plt.suptitle("Example boxes across area-ratio buckets", fontsize=16, y=1.002)
    plt.tight_layout()
    plt.show()

show_bucket_examples(df_valid, bucket_labels, n_per_bucket=3)

### 7.3 Final Outlier Thresholds

In [ ]:
SMALL_AREA_RATIO_THRESH = 0.0005   # box covers < 0.05% of its image
LARGE_AREA_RATIO_THRESH = 0.15     # box covers > 15% of its image

SMALL_MASK = (~NEG_DIM_MASK) & (df["area_ratio"] < SMALL_AREA_RATIO_THRESH)
LARGE_MASK = (~NEG_DIM_MASK) & (df["area_ratio"] > LARGE_AREA_RATIO_THRESH)
NORMAL_MASK = ~(NEG_DIM_MASK | SMALL_MASK | LARGE_MASK)

print(f"Negative/zero-dimension boxes : {NEG_DIM_MASK.sum()}")
print(f"Small outlier boxes (< {SMALL_AREA_RATIO_THRESH:.3%} of image) : {SMALL_MASK.sum()}")
print(f"Large outlier boxes (> {LARGE_AREA_RATIO_THRESH:.1%} of image) : {LARGE_MASK.sum()}")
print(f"Normal boxes                                        : {NORMAL_MASK.sum()}")

plt.figure(figsize=(12, 6))
plt.hist(df.loc[NORMAL_MASK, "area_ratio"] * 100, bins=80, color="steelblue", alpha=0.8, label="normal")
plt.hist(df.loc[SMALL_MASK, "area_ratio"] * 100, bins=20, color="orange", alpha=0.9, label="small outlier")
plt.hist(df.loc[LARGE_MASK, "area_ratio"] * 100, bins=20, color="red", alpha=0.9, label="large outlier")
plt.axvline(SMALL_AREA_RATIO_THRESH * 100, color="orange", linestyle="--", linewidth=1)
plt.axvline(LARGE_AREA_RATIO_THRESH * 100, color="red", linestyle="--", linewidth=1)
plt.yscale("log")
plt.xlabel("Bounding box area as % of its image's area")
plt.ylabel("Number of boxes (log scale)")
plt.title("Bounding box area (relative to image size) with outliers highlighted")
plt.legend()
plt.show()

### 7.4 Visualize Final Small vs Large Outliers

In [ ]:
def show_outlier_columns(small_mask, large_mask, n=5, colors=("blue", "red")):
    def sample_rows(mask, n):
        sub = df.loc[mask]
        ids = sub["image_id"].unique()
        sample_ids = random.sample(list(ids), min(n, len(ids)))
        return [sub[sub["image_id"] == img_id].iloc[0] for img_id in sample_ids]

    small_examples = sample_rows(small_mask, n)
    large_examples = sample_rows(large_mask, n)

    fig, axes = plt.subplots(n, 2, figsize=(12, 6 * n))
    axes = np.atleast_2d(axes)

    for col_i, (examples, mask, color, col_title) in enumerate([
        (small_examples, small_mask, colors[0], "Small-area outlier"),
        (large_examples, large_mask, colors[1], "Large-area outlier"),
    ]):
        for row_i in range(n):
            ax = axes[row_i, col_i]
            if row_i >= len(examples):
                ax.axis("off")
                continue

            ex_row = examples[row_i]
            img_id = ex_row.image_id
            img = np.array(Image.open(TRAIN_IMG_DIR / f"{img_id}.jpg"))
            ax.imshow(img)

            rows = df[(df["image_id"] == img_id) & mask]
            for _, row in rows.iterrows():
                rect = patches.Rectangle(
                    (row.x_min, row.y_min),
                    max(row.box_width, 1), max(row.box_height, 1),
                    linewidth=2.5, edgecolor=color, facecolor="none")
                ax.add_patch(rect)

            ax.set_title(f"{col_title} | {img_id}\n{ex_row.area_ratio:.2%} of image", fontsize=9)
            ax.axis("off")

    plt.tight_layout()
    plt.show()

show_outlier_columns(SMALL_MASK, LARGE_MASK)

## 8. Aspect Ratio Distribution

In [ ]:
valid = df[~NEG_DIM_MASK].copy()
valid["aspect_ratio"] = valid["box_width"] / valid["box_height"]

plt.figure(figsize=(12, 6))
sns.histplot(valid["aspect_ratio"], bins=100, color="teal")
plt.xlim(0, 5)
plt.axvline(1.0, color="black", linestyle="--", label="square (1:1)")
plt.xlabel("Aspect ratio (width / height)")
plt.ylabel("Number of boxes")
plt.title("Bounding box aspect ratio distribution")
plt.legend()
plt.show()

print(valid["aspect_ratio"].describe())

## 9. Extracting and Separating Bounding Box Attributes

Consolidate every box attribute derived above (coordinates, area, aspect ratio,
outlier flags) into a single clean, well-typed dataframe. It is saved once,
at the end of Section 9.1, after the duplicate-box decision below is folded in.

In [ ]:
clean_df = df.copy()
clean_df["x_max"] = clean_df["x_min"] + clean_df["box_width"]
clean_df["y_max"] = clean_df["y_min"] + clean_df["box_height"]
clean_df["area"] = clean_df["box_width"] * clean_df["box_height"]
clean_df["aspect_ratio"] = clean_df["box_width"] / clean_df["box_height"].replace(0, np.nan)

clean_df["is_negative_dim"] = NEG_DIM_MASK
clean_df["is_small_outlier"] = SMALL_MASK
clean_df["is_large_outlier"] = LARGE_MASK
clean_df["is_outlier"] = NEG_DIM_MASK | SMALL_MASK | LARGE_MASK

attribute_cols = [
    "image_id", "width", "height", "source",
    "x_min", "y_min", "box_width", "box_height", "x_max", "y_max",
    "area", "aspect_ratio",
    "is_negative_dim", "is_small_outlier", "is_large_outlier", "is_outlier",
]
clean_df = clean_df[attribute_cols]

display(clean_df.head())
print(f"Outlier boxes flagged: {clean_df['is_outlier'].sum()} / {len(clean_df)}")

### 9.1 Duplicate / Interchangeable Box Detection (IoU)

Two boxes on the same image with very high IoU are essentially duplicate
annotations of the same wheat head. Detected here via pairwise IoU per image.
For each duplicate pair, the **first** box is kept and the **second** is marked
for exclusion — keeping both would let the model be "rewarded twice" for the
same wheat head during training.

In [ ]:
IOU_DUPLICATE_THRESH = 0.65  # boxes overlapping more than this are treated as interchangeable/duplicate

def iou_matrix(x_min, y_min, x_max, y_max):
    areas = (x_max - x_min) * (y_max - y_min)
    xx1 = np.maximum(x_min[:, None], x_min[None, :])
    yy1 = np.maximum(y_min[:, None], y_min[None, :])
    xx2 = np.minimum(x_max[:, None], x_max[None, :])
    yy2 = np.minimum(y_max[:, None], y_max[None, :])
    inter_w = np.clip(xx2 - xx1, 0, None)
    inter_h = np.clip(yy2 - yy1, 0, None)
    inter = inter_w * inter_h
    union = areas[:, None] + areas[None, :] - inter
    return np.where(union > 0, inter / union, 0)

dup_records = []
for image_id, group in clean_df.groupby("image_id"):
    if len(group) < 2:
        continue
    x_min, y_min = group["x_min"].to_numpy(), group["y_min"].to_numpy()
    x_max, y_max = group["x_max"].to_numpy(), group["y_max"].to_numpy()
    ious = iou_matrix(x_min, y_min, x_max, y_max)

    idx_i, idx_j = np.triu_indices(len(group), k=1)
    pair_ious = ious[idx_i, idx_j]
    dup_mask = pair_ious > IOU_DUPLICATE_THRESH

    for i, j, iou_val in zip(idx_i[dup_mask], idx_j[dup_mask], pair_ious[dup_mask]):
        dup_records.append({
            "image_id": image_id,
            "source": group["source"].iloc[0],
            "box_i": group.index[i],
            "box_j": group.index[j],
            "iou": iou_val,
        })

dup_df = pd.DataFrame(dup_records)
print(f"Total interchangeable (duplicate) box pairs at IoU > {IOU_DUPLICATE_THRESH}: {len(dup_df)}")

if len(dup_df) > 0:
    print(f"Images affected: {dup_df['image_id'].nunique()}")

    dup_per_source = dup_df.groupby("source").size().rename("duplicate_pairs")
    dup_per_image = dup_df.groupby("image_id").size().sort_values(ascending=False).rename("duplicate_pairs")

    display(dup_per_source.to_frame())
    display(dup_per_image.head(10).to_frame())

    plt.figure(figsize=(10, 5))
    sns.barplot(x=dup_per_source.index, y=dup_per_source.values,
                hue=dup_per_source.index, palette="rocket", legend=False)
    plt.ylabel("Duplicate box pairs")
    plt.title(f"Interchangeable box pairs per source (IoU > {IOU_DUPLICATE_THRESH})")
    plt.xticks(rotation=45)
    plt.show()
else:
    print("No interchangeable boxes found at this threshold.")

In [ ]:
# --- 9.0.5 IoU Calibration: what do pairs at different overlap levels actually look like? ---

def all_pairwise_ious(boxes_df):
    """Compute IoU for every same-image box pair, return as a flat dataframe."""
    records = []
    for image_id, group in boxes_df.groupby("image_id"):
        if len(group) < 2:
            continue
        x_min, y_min = group["x_min"].to_numpy(), group["y_min"].to_numpy()
        x_max, y_max = group["x_max"].to_numpy(), group["y_max"].to_numpy()
        ious = iou_matrix(x_min, y_min, x_max, y_max)
        idx_i, idx_j = np.triu_indices(len(group), k=1)
        for i, j, v in zip(idx_i, idx_j, ious[idx_i, idx_j]):
            if v > 0:  # skip completely non-overlapping pairs
                records.append({
                    "image_id": image_id,
                    "box_i": group.index[i], "box_j": group.index[j], "iou": v,
                })
    return pd.DataFrame(records)

# clean_df must already have x_max/y_max (from Section 9) before this runs
all_pairs_df = all_pairwise_ious(clean_df)
print(f"Total overlapping box pairs found: {len(all_pairs_df)}")

iou_bands = [(0.3, 0.4), (0.4, 0.5), (0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 1.01)]
n_per_band = 3

fig, axes = plt.subplots(len(iou_bands), n_per_band, figsize=(5 * n_per_band, 5 * len(iou_bands)))

for row_i, (lo, hi) in enumerate(iou_bands):
    band_df = all_pairs_df[(all_pairs_df["iou"] >= lo) & (all_pairs_df["iou"] < hi)]
    sampled = band_df.sample(min(n_per_band, len(band_df)), random_state=RANDOM_SEED) if len(band_df) else band_df

    for col_i in range(n_per_band):
        ax = axes[row_i, col_i]
        ax.axis("off")
        if col_i >= len(sampled):
            if col_i == 0 and len(sampled) == 0:
                ax.text(0.5, 0.5, f"No pairs in [{lo:.1f}, {hi:.1f})", ha="center", va="center")
            continue

        rec = sampled.iloc[col_i]
        img = np.array(Image.open(TRAIN_IMG_DIR / f"{rec.image_id}.jpg"))
        ax.imshow(img)
        for box_idx, color in [(rec.box_i, "red"), (rec.box_j, "cyan")]:
            b = clean_df.loc[box_idx]
            rect = patches.Rectangle((b.x_min, b.y_min), b.box_width, b.box_height,
                                      linewidth=2.5, edgecolor=color, facecolor="none")
            ax.add_patch(rect)
        ax.set_title(f"IoU=[{lo:.1f},{hi:.1f}) | {rec.image_id}\nactual={rec.iou:.2f}", fontsize=9)

plt.suptitle("Box-pair overlap calibration: same head (duplicate) vs. two real, touching heads?", fontsize=15, y=1.001)
plt.tight_layout()
plt.show()

# Quick distribution check to see how many pairs fall in the ambiguous zone
print(all_pairs_df["iou"].describe())
print("\nPairs per band:")
for lo, hi in iou_bands:
    n = ((all_pairs_df["iou"] >= lo) & (all_pairs_df["iou"] < hi)).sum()
    print(f"  [{lo:.1f}, {hi:.1f}): {n}")

In [ ]:

clean_df["use_for_training"] = ~(clean_df["is_outlier"])

clean_csv_path = OUTPUT_DIR / "train_boxes_clean.csv"
clean_df.to_csv(clean_csv_path, index=False)

print(f"Saved cleaned, attribute-separated data to: {clean_csv_path.resolve()}")
print(f"Outlier boxes flagged             : {clean_df['is_outlier'].sum()} / {len(clean_df)}")
print(f"Boxes usable for training         : {clean_df['use_for_training'].sum()} / {len(clean_df)}")

## 10. Train / Validation Split (Source-Stratified)

The split is done **per image** (not per box) and **stratified by source**, since
box density and visual characteristics vary noticeably by source (Section 6).
Images with zero boxes are grouped into their own pseudo-source ("no_box") so
they are distributed proportionally across train/val too, rather than landing
disproportionately in one split.

In [ ]:
image_source = (
    df.groupby("image_id")["source"]
    .first()
    .reindex(all_train_images)
    .fillna("no_box")
)

image_level_df = (
    image_source.rename("source")
    .to_frame()
    .reset_index()
    .rename(columns={"index": "image_id"})
)

RANDOM_SEED = 42

# First split: 80% train, 20% temporary
train_ids, temp_ids = train_test_split(
    image_level_df["image_id"],
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=image_level_df["source"],
)

# Second split: divide temporary set into 10% validation and 10% test
temp_sources = image_level_df.set_index("image_id").loc[temp_ids, "source"]

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=temp_sources,
)

train_ids = set(train_ids)
val_ids = set(val_ids)
test_ids = set(test_ids)

image_level_df["split"] = image_level_df["image_id"].apply(
    lambda image_id: (
        "train" if image_id in train_ids
        else "val" if image_id in val_ids
        else "test"
    )
)

print(image_level_df["split"].value_counts())
display(pd.crosstab(image_level_df["source"], image_level_df["split"]))

split_csv_path = OUTPUT_DIR / "image_split.csv"
image_level_df.to_csv(split_csv_path, index=False)

print(f"Saved split assignment to: {split_csv_path.resolve()}")

split_lookup = image_level_df.set_index("image_id")["split"]

## 12. Convert to YOLO-Format Dataset

Builds a split-aware `images/{train,val}` + `labels/{train,val}` YOLO dataset
(single class: `wheat`):
- One `.txt` label file per image (empty file for images with no usable boxes)
- Each line: `class x_center y_center width height` (all normalized 0-1)
- Boxes excluded from labels: outliers (Section 7) **and** the dropped copy of each duplicate pair (Section 9.1) — i.e. only `use_for_training == True` boxes are written
- Images are written into `train/` or `val/` per the Section 10 split assignment

The augmentation pipeline from Section 11 is **not** applied here — it stays as
a pipeline object the Week 3/4 training script imports and runs on-the-fly,
train split only.

In [ ]:
from sklearn.model_selection import train_test_split

# --- Build an 80/10/10 split, stratified by source ---
image_source_df = clean_df[["image_id", "source"]].drop_duplicates()

train_ids, temp_ids = train_test_split(
    image_source_df["image_id"], test_size=0.20,
    stratify=image_source_df["source"], random_state=RANDOM_SEED,
)
temp_source = image_source_df.set_index("image_id").loc[temp_ids, "source"]
val_ids, test_ids = train_test_split(
    temp_ids, test_size=0.50, stratify=temp_source, random_state=RANDOM_SEED,
)  # 0.5 of the 20% held out -> 10% val, 10% test

# images with no boxes have no known source - split them the same way, unstratified
no_box_ids = sorted(set(all_train_images) - set(image_source_df["image_id"]))
nb_train, nb_temp = train_test_split(no_box_ids, test_size=0.20, random_state=RANDOM_SEED)
nb_val, nb_test = train_test_split(nb_temp, test_size=0.50, random_state=RANDOM_SEED)

split_lookup = {}
for img_id in list(train_ids) + nb_train:
    split_lookup[img_id] = "train"
for img_id in list(val_ids) + nb_val:
    split_lookup[img_id] = "val"
for img_id in list(test_ids) + nb_test:
    split_lookup[img_id] = "test"

print(pd.Series(split_lookup).value_counts())

In [ ]:
YOLO_IMG_DIR = {s: YOLO_DIR / "images" / s for s in ["train", "val", "test"]}
YOLO_LBL_DIR = {s: YOLO_DIR / "labels" / s for s in ["train", "val", "test"]}
for d in list(YOLO_IMG_DIR.values()) + list(YOLO_LBL_DIR.values()):
    d.mkdir(parents=True, exist_ok=True)

def to_yolo_line(row, img_w, img_h):
    x_center = (row.x_min + row.box_width / 2) / img_w
    y_center = (row.y_min + row.box_height / 2) / img_h
    w = row.box_width / img_w
    h = row.box_height / img_h
    x_center, y_center, w, h = (float(np.clip(v, 0, 1)) for v in (x_center, y_center, w, h))
    return f"{CLASS_ID} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}"

boxes_for_yolo = clean_df[clean_df["use_for_training"]]

for img_id in all_train_images:
    split = split_lookup.get(img_id, "train")  # safe now: "train"/"val"/"test" all exist as keys
    rows = boxes_for_yolo[boxes_for_yolo["image_id"] == img_id]

    img_w, img_h = 1024, 1024
    if len(rows) > 0:
        img_w = int(rows.iloc[0]["width"])
        img_h = int(rows.iloc[0]["height"])

    lines = [to_yolo_line(r, img_w, img_h) for r in rows.itertuples()]
    (YOLO_LBL_DIR[split] / f"{img_id}.txt").write_text("\n".join(lines))

    src_img = TRAIN_IMG_DIR / f"{img_id}.jpg"
    dst_img = YOLO_IMG_DIR[split] / f"{img_id}.jpg"
    if not dst_img.exists():
        try:
            os.link(src_img, dst_img)
        except OSError:
            shutil.copy(src_img, dst_img)

data_yaml = f"""path: {YOLO_DIR.resolve()}
train: images/train
val: images/val
test: images/test
names:
  0: {CLASS_NAME}
"""
(YOLO_DIR / "data.yaml").write_text(data_yaml)

for split in ["train", "val", "test"]:
    n_img = len(list(YOLO_IMG_DIR[split].glob("*.jpg")))
    n_lbl = len(list(YOLO_LBL_DIR[split].glob("*.txt")))
    print(f"{split:5s}: {n_img} images, {n_lbl} labels")

## 13. YOLO Label Sanity Check

In [ ]:
def verify_yolo_label(image_id):
    split = split_lookup.get(image_id, "train")
    img = np.array(Image.open(TRAIN_IMG_DIR / f"{image_id}.jpg"))
    h, w = img.shape[:2]

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    # ---- Left: original COCO-style representation [x_min, y_min, w, h] ----
    coco_rows = boxes_for_yolo[boxes_for_yolo["image_id"] == image_id]
    axes[0].imshow(img)
    for _, row in coco_rows.iterrows():
        rect = patches.Rectangle((row.x_min, row.y_min), row.box_width, row.box_height,
                                  linewidth=2, edgecolor="red", facecolor="none")
        axes[0].add_patch(rect)
    axes[0].set_title(f"COCO-style (original)\n{image_id}  |  split={split}  |  {len(coco_rows)} boxes")
    axes[0].axis("off")

    # ---- Right: re-drawn from the written YOLO label file ----
    lines = (YOLO_LBL_DIR[split] / f"{image_id}.txt").read_text().splitlines()
    axes[1].imshow(img)
    for line in lines:
        if not line.strip():
            continue
        cls, xc, yc, bw, bh = map(float, line.split())
        x_min = (xc - bw / 2) * w
        y_min = (yc - bh / 2) * h
        rect = patches.Rectangle((x_min, y_min), bw * w, bh * h,
                                  linewidth=2, edgecolor="lime", facecolor="none")
        axes[1].add_patch(rect)
    axes[1].set_title(f"YOLO-format (reconstructed)\n{image_id}  |  {len(lines)} boxes")
    axes[1].axis("off")

    plt.suptitle("COCO vs YOLO representation — sanity check")
    plt.tight_layout()
    plt.show()

check_id = next(iter(boxes_for_yolo["image_id"].unique()))
verify_yolo_label(check_id)

In [ ]:
import os
print(os.path.exists("/kaggle/working/outputs/yolo_dataset/data.yaml"))
print(os.listdir("/kaggle/working") if os.path.exists("/kaggle/working") else "working dir empty/missing")

In [ ]:
!pip install -U ultralytics

from ultralytics import YOLO

model = YOLO("yolo11s.pt")

results = model.train(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="global_wheat",
    name="baseline_yolo11s"
)

In [ ]:
metrics = model.val(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    imgsz=640
)

print("\n" + "="*50)
print("           VALIDATION RESULTS")
print("="*50)

print(f"mAP@50       : {metrics.box.map50:.4f}")
print(f"mAP@50-95    : {metrics.box.map:.4f}")
print(f"Precision    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")

print("="*50)

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Paths
val_images = "/kaggle/working/outputs/yolo_dataset/images/val"
val_labels = "/kaggle/working/outputs/yolo_dataset/labels/val"

# ------------------------------------------------
# Helper function to calculate IoU
# ------------------------------------------------
def calculate_iou(box1, box2):
    """box format: [x1, y1, x2, y2]"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0

# ------------------------------------------------
# 1. Scan all images to evaluate predictions
# ------------------------------------------------
print("Scanning validation set to find best and worst examples...")
evaluated_images = []

image_files = [
    f for f in os.listdir(val_images)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

for image_file in image_files:
    image_path = os.path.join(val_images, image_file)
    label_path = os.path.join(val_labels, os.path.splitext(image_file)[0] + ".txt")
    
    image = cv2.imread(image_path)
    if image is None:
        continue
    h, w = image.shape[:2]
    
    # Load Ground Truth boxes
    gt_boxes = []
    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            for line in f.readlines():
                cls, x_center, y_center, box_w, box_h = map(float, line.strip().split())
                x1 = (x_center - box_w / 2) * w
                y1 = (y_center - box_h / 2) * h
                x2 = (x_center + box_w / 2) * w
                y2 = (y_center + box_h / 2) * h
                gt_boxes.append([x1, y1, x2, y2])
    
    # Get Model Predictions
    results = model.predict(source=image_path, imgsz=640, verbose=False, conf=0.25)
    pred_boxes = []
    confidences = []
    
    if len(results[0].boxes) > 0:
        for box in results[0].boxes:
            x1, y1, x2, y2 = map(float, box.xyxy[0])
            conf = float(box.conf[0])
            pred_boxes.append([x1, y1, x2, y2])
            confidences.append(conf)
            
    # Calculate Errors (False Positives & False Negatives)
    errors = 0
    
    # Check for False Negatives (GT exists, but no good prediction)
    for gt in gt_boxes:
        best_iou = max([calculate_iou(gt, p) for p in pred_boxes], default=0)
        if best_iou < 0.5:
            errors += 1  # Missed object
            
    # Check for False Positives (Prediction exists, but no matching GT)
    for p in pred_boxes:
        best_iou = max([calculate_iou(p, gt) for gt in gt_boxes], default=0)
        if best_iou < 0.5:
            errors += 1  # Hallucinated object
            
    # Save the evaluated image data
    evaluated_images.append({
        "file": image_file,
        "path": image_path,
        "label_path": label_path,
        "errors": errors,
        "gt_boxes": gt_boxes,
        "pred_boxes": pred_boxes,
        "confidences": confidences
    })

# ------------------------------------------------
# 2. Sort and select the 2 Best and 2 Worst
# ------------------------------------------------
# Sort ascending by errors (0 errors first, highest errors last)
evaluated_images.sort(key=lambda x: x["errors"])

# Pick the 2 with the LEAST errors (Good examples)
good_examples = evaluated_images[:2]

# Pick the 2 with the MOST errors (Bad examples)
bad_examples = evaluated_images[-2:]

print(f"Total images scanned: {len(evaluated_images)}")
print(f"Best examples have {good_examples[0]['errors']} and {good_examples[1]['errors']} errors.")
print(f"Worst examples have {bad_examples[0]['errors']} and {bad_examples[1]['errors']} errors.")

# Combine them to display (Good first, then Bad)
examples_to_show = [
    (good_examples[0], "GOOD Example 1"),
    (good_examples[1], "GOOD Example 2"),
    (bad_examples[0], "BAD Example 1"),
    (bad_examples[1], "BAD Example 2")
]

# ------------------------------------------------
# 3. Display the 4 Examples
# ------------------------------------------------
for idx, (prob, title) in enumerate(examples_to_show):
    image_path = prob["path"]
    label_path = prob["label_path"]
    
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]
    
    # --- Ground Truth Image ---
    gt_image = image.copy()
    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            labels = f.readlines()
        for line in labels:
            cls, x_center, y_center, box_w, box_h = map(float, line.strip().split())
            x_center *= w; y_center *= h; box_w *= w; box_h *= h
            x1 = int(x_center - box_w / 2); y1 = int(y_center - box_h / 2)
            x2 = int(x_center + box_w / 2); y2 = int(y_center + box_h / 2)
            cv2.rectangle(gt_image, (x1, y1), (x2, y2), (0, 255, 0), 2)

    # --- Prediction Image (Using stored data for speed) ---
    pred_image = image.copy()
    for i, box in enumerate(prob["pred_boxes"]):
        x1, y1, x2, y2 = map(int, box)
        confidence = prob["confidences"][i]
        
        # Color code: Red for low confidence, Blue for high confidence
        color = (255, 0, 0) if confidence < 0.5 else (0, 0, 255)
        
        cv2.rectangle(pred_image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(pred_image, f"{confidence:.2f}", (x1, max(y1 - 5, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # --- Plotting ---
    plt.figure(figsize=(16, 7))
    
    plt.subplot(1, 2, 1)
    plt.imshow(gt_image)
    plt.title(f"Ground Truth\n({title} | {prob['errors']} errors)")
    plt.axis("off")
    
    plt.subplot(1, 2, 2)
    plt.imshow(pred_image)
    plt.title("Model Predictions\n(Red=Low Conf, Blue=High Conf)")
    plt.axis("off")
    
    plt.tight_layout()
    plt.show()

# Comparison between YOLO11 (n/s/m)

In [ ]:
from ultralytics import YOLO

models_to_compare = {
    "YOLO11n": "yolo11n.pt",   # Fastest, smallest
    "YOLO11s": "yolo11s.pt",   # Your current baseline ✅
    "YOLO11m": "yolo11m.pt",   # More accurate
}

results = {}

for name, model_path in models_to_compare.items():
    print(f"\n{'='*60}")
    print(f"Training {name}...")
    print(f"{'='*60}\n")
    
    model = YOLO(model_path)
    
    results[name] = model.train(
        data="/kaggle/working/outputs/yolo_dataset/data.yaml",
        epochs=30,
        imgsz=640,
        batch=16,
        project="global_wheat_week2",
        name=f"compare_{name.lower()}",
        verbose=False
    )

In [ ]:
validation_metrics = {}
for name in models_to_compare.keys():
    print(f"\nEvaluating {name}...")

    folder_suffix = f"{name.lower()}-2" if name == "YOLO11n" else name.lower()
    model = YOLO(f"/kaggle/working/runs/detect/global_wheat_week2/compare_{folder_suffix}/weights/best.pt")

    metrics = model.val(
        data="/kaggle/working/outputs/yolo_dataset/data.yaml",
        imgsz=640,
        verbose=False
    )

    validation_metrics[name] = {
        "mAP@50": metrics.box.map50,
        "mAP@50-95": metrics.box.map,
        "Precision": metrics.box.mp,
        "Recall": metrics.box.mr,
    }

    print(f"  mAP@50:    {metrics.box.map50:.4f}")
    print(f"  mAP@50-95: {metrics.box.map:.4f}")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall:    {metrics.box.mr:.4f}")

# Create comparison table
comparison_df = pd.DataFrame(validation_metrics).T
print("\n" + "="*70)
print("           MODEL COMPARISON (Validation Set)")
print("="*70)
display(comparison_df)

In [ ]:
# Create comparison plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics_to_plot = ["mAP@50", "mAP@50-95", "Precision", "Recall"]
colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#FFA07A"]

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]
    values = [validation_metrics[model][metric] for model in validation_metrics.keys()]
    
    bars = ax.bar(validation_metrics.keys(), values, color=colors[idx])
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} Comparison")
    ax.set_ylim(0, 1)
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle("YOLO Model Comparison on Validation Set", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import os
import pandas as pd

print("\n" + "="*70)
print("           TRAINING TIME & MODEL SIZE")
print("="*70)

for name in models_to_compare.keys():
    # CORRECTED PATH - added runs/detect/
    folder_suffix = f"{name.lower()}-2" if name == "YOLO11n" else name.lower()
    model_dir = f"runs/detect/global_wheat_week2/compare_{folder_suffix}"
    best_model = f"{model_dir}/weights/best.pt"
    
    if os.path.exists(best_model):
        size_mb = os.path.getsize(best_model) / (1024 * 1024)
        
        try:
            results_csv = f"{model_dir}/results.csv"
            if os.path.exists(results_csv):
                df = pd.read_csv(results_csv)
                epochs_trained = len(df)
                
                # Calculate total training time from the 'time' column (seconds per epoch)
                if 'time' in df.columns:
                    total_time_seconds = df['time'].sum()
                    total_time_minutes = total_time_seconds / 60
                    time_str = f"{total_time_minutes:.2f} min"
                else:
                    time_str = "N/A (time column not found)"
                    
                print(f"{name:10s} | Size: {size_mb:6.2f} MB | Epochs: {epochs_trained} | Time: {time_str}")
            else:
                print(f"{name:10s} | Results.csv not found at {results_csv}")
        except Exception as e:
            print(f"{name:10s} | Error reading results: {e}")
    else:
        print(f"{name:10s} | Model not found at {best_model}")

In [ ]:
# ============================================================
# Evaluate ALL THREE models on the held-out TEST set
# ============================================================

test_metrics_all = {}

for name in models_to_compare.keys():
    print(f"\nEvaluating {name} on TEST set...")

    folder_suffix = f"{name.lower()}-2" if name == "YOLO11n" else name.lower()
    model = YOLO(f"/kaggle/working/runs/detect/global_wheat_week2/compare_{folder_suffix}/weights/best.pt")

    test_metrics = model.val(
        data="/kaggle/working/outputs/yolo_dataset/data.yaml",
        split="test",       # <-- evaluates on images/test + labels/test, not val
        imgsz=640,
        verbose=False,
    )

    test_metrics_all[name] = {
        "mAP@50": test_metrics.box.map50,
        "mAP@50-95": test_metrics.box.map,
        "Precision": test_metrics.box.mp,
        "Recall": test_metrics.box.mr,
    }

    print(f"  mAP@50:    {test_metrics.box.map50:.4f}")
    print(f"  mAP@50-95: {test_metrics.box.map:.4f}")
    print(f"  Precision: {test_metrics.box.mp:.4f}")
    print(f"  Recall:    {test_metrics.box.mr:.4f}")

test_comparison_df = pd.DataFrame(test_metrics_all).T
print("\n" + "=" * 70)
print("           MODEL COMPARISON (TEST Set)")
print("=" * 70)
display(test_comparison_df)

In [ ]:
# ============================================================
# Visualize predictions vs ground truth on sample TEST images
# — for all three models
# ============================================================
TEST_IMG_DIR_YOLO = YOLO_DIR / "images" / "test"
TEST_LBL_DIR_YOLO = YOLO_DIR / "labels" / "test"

def compare_gt_vs_prediction(model, model_label, image_id, img_dir=TEST_IMG_DIR_YOLO, lbl_dir=TEST_LBL_DIR_YOLO,
                              conf=0.25, ax_pair=None):
    img_path = img_dir / f"{image_id}.jpg"
    img = np.array(Image.open(img_path))
    h, w = img.shape[:2]

    gt_lines = (lbl_dir / f"{image_id}.txt").read_text().splitlines()
    gt_boxes = []
    for line in gt_lines:
        if not line.strip():
            continue
        _, xc, yc, bw, bh = map(float, line.split())
        x_min = (xc - bw / 2) * w
        y_min = (yc - bh / 2) * h
        gt_boxes.append((x_min, y_min, bw * w, bh * h))

    result = model.predict(source=str(img_path), conf=conf, imgsz=640, verbose=False)[0]
    pred_boxes = result.boxes.xyxy.cpu().numpy()
    pred_scores = result.boxes.conf.cpu().numpy()

    if ax_pair is None:
        fig, ax_pair = plt.subplots(1, 2, figsize=(14, 7))
    ax_gt, ax_pred = ax_pair

    ax_gt.imshow(img)
    for x, y, bw, bh in gt_boxes:
        rect = patches.Rectangle((x, y), bw, bh, linewidth=2, edgecolor="lime", facecolor="none")
        ax_gt.add_patch(rect)
    ax_gt.set_title(f"Ground truth | {image_id}\n{len(gt_boxes)} boxes")
    ax_gt.axis("off")

    ax_pred.imshow(img)
    for (x1, y1, x2, y2), score in zip(pred_boxes, pred_scores):
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="red", facecolor="none")
        ax_pred.add_patch(rect)
    ax_pred.set_title(f"{model_label} predictions (conf > {conf})\n{len(pred_boxes)} boxes")
    ax_pred.axis("off")

def show_test_predictions_grid(model, model_label, n=4, seed=RANDOM_SEED):
    test_ids = [p.stem for p in TEST_IMG_DIR_YOLO.glob("*.jpg")]
    sample_ids = random.Random(seed).sample(test_ids, min(n, len(test_ids)))

    fig, axes = plt.subplots(n, 2, figsize=(14, 7 * n))
    axes = np.atleast_2d(axes)

    for row_i, image_id in enumerate(sample_ids):
        compare_gt_vs_prediction(model, model_label, image_id, ax_pair=axes[row_i])

    plt.suptitle(f"Ground truth vs. {model_label} predictions — test set", fontsize=16, y=1.001)
    plt.tight_layout()
    plt.show()

for name in models_to_compare.keys():
    folder_suffix = f"{name.lower()}-2" if name.lower() == "yolo11n" else name.lower()
    loop_model = YOLO(f"/kaggle/working/runs/detect/global_wheat_week2/compare_{folder_suffix}/weights/best.pt")
    show_test_predictions_grid(loop_model, name, n=4)

# Faster R-CNN

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import torchvision.transforms.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Load your saved splits and clean data
split_df = pd.read_csv(OUTPUT_DIR / "image_split.csv")
clean_df = pd.read_csv(OUTPUT_DIR / "train_boxes_clean.csv")

train_ids = split_df[split_df['split'] == 'train']['image_id'].tolist()
val_ids = split_df[split_df['split'] == 'val']['image_id'].tolist()

train_boxes = clean_df[clean_df['use_for_training'] == True]
val_boxes = clean_df[clean_df['use_for_training'] == True]

# Simpler, faster transforms
def get_transform(train):
    if train:
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            ToTensorV2()
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))
    else:
        return A.Compose([
            A.HorizontalFlip(p=0.0),
            ToTensorV2()
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

class WheatDataset(Dataset):
    def __init__(self, image_ids, boxes_df, image_dir, transforms=None):
        self.image_ids = image_ids
        self.boxes_df = boxes_df
        self.image_dir = image_dir
        self.transforms = transforms

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_path = self.image_dir / f"{img_id}.jpg"
        image = np.array(Image.open(img_path).convert("RGB"))
        
        records = self.boxes_df[self.boxes_df['image_id'] == img_id]
        
        boxes, areas, labels = [], [], []
        for _, row in records.iterrows():
            boxes.append([row['x_min'], row['y_min'], row['x_max'], row['y_max']])
            areas.append(row['area'])
            labels.append(1)
            
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            areas = torch.zeros((0,), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            areas = torch.tensor(areas, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": areas,
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64)
        }
        
        if self.transforms:
            transformed = self.transforms(
                image=image, 
                bboxes=target['boxes'].numpy(), 
                labels=target['labels'].numpy()
            )
            image = transformed['image']
            
            if isinstance(image, np.ndarray):
                image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            elif image.dtype == torch.uint8:
                image = image.float() / 255.0
            
            # Resize to 640x640
            image = F.resize(image, [640, 640])
            
            if target['boxes'].numel() > 0:
                orig_h, orig_w = 1024, 1024
                scale_x = 640 / orig_w
                scale_y = 640 / orig_h
                target['boxes'][:, [0, 2]] *= scale_x
                target['boxes'][:, [1, 3]] *= scale_y
                target['area'] = (target['boxes'][:, 2] - target['boxes'][:, 0]) * \
                                 (target['boxes'][:, 3] - target['boxes'][:, 1])
            
            target['boxes'] = torch.tensor(transformed['bboxes'], dtype=torch.float32)
            if target['boxes'].numel() > 0:
                target['boxes'][:, [0, 2]] *= scale_x
                target['boxes'][:, [1, 3]] *= scale_y
            
        return image, target

# ===== SPEED OPTIMIZATION 1: num_workers=2 (Kaggle-friendly) =====
dataset_train = WheatDataset(train_ids, train_boxes, TRAIN_IMG_DIR, get_transform(train=True))
dataset_val = WheatDataset(val_ids, val_boxes, TRAIN_IMG_DIR, get_transform(train=False))

data_loader_train = DataLoader(
    dataset_train, batch_size=8, shuffle=True, num_workers=2,  # batch=8, workers=2
    collate_fn=lambda x: tuple(zip(*x))
)
data_loader_val = DataLoader(
    dataset_val, batch_size=8, shuffle=False, num_workers=2,
    collate_fn=lambda x: tuple(zip(*x))
)

print(f"Train: {len(dataset_train)} images, Val: {len(dataset_val)} images")

In [ ]:
import torchvision
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

model = fasterrcnn_mobilenet_v3_large_fpn(weights=FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=2)
model.to(device)

print("Model loaded and ready")

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.01, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

from torch.amp import autocast, GradScaler
scaler = GradScaler()

num_epochs = 25

print("Starting FAST Faster R-CNN Training (MobileNetV3 + FP16 + 640px)...")
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    
    for images, targets in data_loader_train:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        optimizer.zero_grad()
        
        # Mixed Precision Forward Pass
        with autocast(device_type='cuda', dtype=torch.float16):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
        
        # Mixed Precision Backward Pass
        scaler.scale(losses).backward()
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += losses.item()
        
    lr_scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {epoch_loss:.4f}")

print("Training Complete!")

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

model.eval()
metric = MeanAveragePrecision(iou_type="bbox", box_format="xyxy")

print("Evaluating on Validation Set...")
with torch.no_grad():
    for images, targets in data_loader_val:
        images = list(img.to(device) for img in images)
        
        preds = model(images)
        
        # Format predictions for torchmetrics
        preds = [{
            "boxes": p["boxes"].cpu(),
            "scores": p["scores"].cpu(),
            "labels": p["labels"].cpu()
        } for p in preds]
        
        # Format targets
        targets = [{
            "boxes": t["boxes"].cpu(),
            "labels": t["labels"].cpu()
        } for t in targets]
        
        metric.update(preds, targets)

# Compute final metrics
metrics = metric.compute()
print("\n" + "="*50)
print("       FASTER R-CNN VALIDATION RESULTS")
print("="*50)
print(f"mAP@50       : {metrics['map_50'].item():.4f}")
print(f"mAP@50-95    : {metrics['map'].item():.4f}")
print("="*50)

In [ ]:
# ------------------------------------------------
# Helper function to calculate IoU
# ------------------------------------------------
def calculate_iou(box1, box2):
    """box format: [x1, y1, x2, y2]"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0

# ------------------------------------------------
# 4. Calculate Precision, Recall, and F1 at IoU=0.5
# ------------------------------------------------
# This matches the way YOLO calculates its global Precision and Recall metrics
iou_threshold = 0.5
conf_threshold = 0.3  # Standard baseline confidence threshold

TP = 0  # True Positives (Correct detections)
FP = 0  # False Positives (Hallucinations)
FN = 0  # False Negatives (Missed objects)

print("Calculating Precision & Recall on Validation Set...")
model.eval()
with torch.no_grad():
    for images, targets in data_loader_val:
        images = list(img.to(device) for img in images)
        preds = model(images)
        
        for pred, target in zip(preds, targets):
            pred_boxes = pred['boxes'].cpu().numpy()
            pred_scores = pred['scores'].cpu().numpy()
            gt_boxes = target['boxes'].cpu().numpy()
            
            # Filter predictions by confidence
            keep = pred_scores >= conf_threshold
            pred_boxes = pred_boxes[keep]
            
            matched_gt_indices = set()
            
            # Evaluate each prediction
            for p_box in pred_boxes:
                best_iou = 0
                best_gt_idx = -1
                
                # Find the best matching Ground Truth
                for idx, g_box in enumerate(gt_boxes):
                    if idx in matched_gt_indices:
                        continue
                    iou = calculate_iou(p_box, g_box) 
                    if iou > best_iou:
                        best_iou = iou
                        best_gt_idx = idx
                
                # If match is good enough, it's a True Positive
                if best_iou >= iou_threshold:
                    TP += 1
                    matched_gt_indices.add(best_gt_idx)
                else:
                    FP += 1  # False Positive
                    
            # Any unmatched Ground Truths are False Negatives
            FN += len(gt_boxes) - len(matched_gt_indices)

# Calculate final metrics
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n" + "="*50)
print("   FASTER R-CNN PRECISION & RECALL (IoU=0.5)")
print("="*50)
print(f"True Positives (TP)  : {TP}")
print(f"False Positives (FP) : {FP}")
print(f"False Negatives (FN) : {FN}")
print("-" * 50)
print(f"Precision            : {precision:.4f}")
print(f"Recall               : {recall:.4f}")
print(f"F1-Score             : {f1_score:.4f}")
print("="*50)

In [ ]:
import cv2

# Re-use your evaluation logic to find best/worst
evaluated_images = []
val_images_dir = TRAIN_IMG_DIR # Since we didn't move images to a YOLO folder

for img_id in val_ids:
    img_path = val_images_dir / f"{img_id}.jpg"
    image = cv2.imread(str(img_path))
    if image is None: continue
    h, w = image.shape[:2]
    
    # Ground Truth
    gt_records = val_boxes[val_boxes['image_id'] == img_id]
    gt_boxes = [[r['x_min'], r['y_min'], r['x_max'], r['y_max']] for _, r in gt_records.iterrows()]
    
    # Predictions
    img_tensor = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
    with torch.no_grad():
        pred = model([img_tensor.to(device)])[0]
        
    pred_boxes = pred['boxes'].cpu().numpy()
    pred_scores = pred['scores'].cpu().numpy()
    
    # Filter by confidence
    keep = pred_scores > 0.3
    pred_boxes = pred_boxes[keep]
    pred_scores = pred_scores[keep]
    
    # Calculate errors (IoU < 0.5)
    errors = 0
    for gt in gt_boxes:
        best_iou = max([calculate_iou(gt, p) for p in pred_boxes], default=0)
        if best_iou < 0.5: errors += 1
    for p in pred_boxes:
        best_iou = max([calculate_iou(p, gt) for gt in gt_boxes], default=0)
        if best_iou < 0.5: errors += 1
        
    evaluated_images.append({"id": img_id, "errors": errors, "gt": gt_boxes, "pred": pred_boxes, "scores": pred_scores})

evaluated_images.sort(key=lambda x: x["errors"])
examples_to_show = [
    (evaluated_images[0], "FR GOOD Example 1"),
    (evaluated_images[1], "FR GOOD Example 2"),
    (evaluated_images[-1], "FR BAD Example 1"),
    (evaluated_images[-2], "FR BAD Example 2")
]

for prob, title in examples_to_show:
    img_path = val_images_dir / f"{prob['id']}.jpg"
    image = cv2.imread(str(img_path))
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    gt_image = image_rgb.copy()
    for box in prob['gt']:
        cv2.rectangle(gt_image, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (0, 255, 0), 2)
        
    pred_image = image_rgb.copy()
    for box, conf in zip(prob['pred'], prob['scores']):
        color = (255, 0, 0) if conf < 0.5 else (0, 0, 255)
        cv2.rectangle(pred_image, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), color, 2)
        cv2.putText(pred_image, f"{conf:.2f}", (int(box[0]), int(box[1])-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        
    plt.figure(figsize=(16, 7))
    plt.subplot(1, 2, 1); plt.imshow(gt_image); plt.title(f"Ground Truth\n({title} | {prob['errors']} errors)"); plt.axis("off")
    plt.subplot(1, 2, 2); plt.imshow(pred_image); plt.title("Faster R-CNN Predictions\n(Red=Low Conf, Blue=High Conf)"); plt.axis("off")
    plt.tight_layout(); plt.show()

# Deformable DETR

## CSV-to-JSON COCO-format converter

In [ ]:
"""
Convert the Global Wheat Detection train.csv into COCO-format JSON
(train/valid split) for use with mmdetection / DINO.

CSV columns expected: image_id, width, height, bbox, source
  bbox is a string like "[x_min, y_min, w, h]" in absolute pixels
  (already COCO-style — no normalization needed).

Just paste this whole cell into your notebook, then in the NEXT cell run:
    convert(
        csv_path="/kaggle/input/global-wheat-detection/train.csv",
        out_dir="/kaggle/working/outputs/coco_dataset",
        val_fraction=0.1,
    )
"""

import ast
import json
import os

import numpy as np
import pandas as pd

CATEGORY = {"id": 0, "name": "wheat_head", "supercategory": "none"}


def build_split(df: pd.DataFrame, image_ids: list) -> dict:
    """Build a COCO dict for the given subset of image_ids."""
    images, annotations = [], []
    img_id_map = {}
    ann_id = 1

    sub = df[df["image_id"].isin(image_ids)]

    for new_id, image_id in enumerate(image_ids, start=1):
        img_id_map[image_id] = new_id
        row0 = sub[sub["image_id"] == image_id].iloc[0]
        images.append(
            {
                "id": new_id,
                "file_name": f"{image_id}.jpg",
                "width": int(row0["width"]),
                "height": int(row0["height"]),
            }
        )

    for _, row in sub.iterrows():
        x, y, w, h = row["bbox"]
        annotations.append(
            {
                "id": ann_id,
                "image_id": img_id_map[row["image_id"]],
                "category_id": CATEGORY["id"],
                "bbox": [round(x, 2), round(y, 2), round(w, 2), round(h, 2)],
                "area": round(w * h, 2),
                "iscrowd": 0,
            }
        )
        ann_id += 1

    return {"images": images, "annotations": annotations, "categories": [CATEGORY]}


def convert(
    csv_path: str,
    out_dir: str,
    val_fraction: float = 0.1,
    seed: int = 42,
    stratify_by_source: bool = False,
) -> None:
    """Convert train.csv into COCO train.json / valid.json.

    Args:
        csv_path: path to the Global Wheat Detection train.csv
        out_dir: folder to write train.json / valid.json into (created if missing)
        val_fraction: fraction of images held out for validation
        seed: random seed for the split (same seed -> same split every run)
        stratify_by_source: if True, splits within each `source` group so
            train/valid keep the same mix of collection sites
    """
    os.makedirs(out_dir, exist_ok=True)

    df = pd.read_csv(csv_path)
    df["bbox"] = df["bbox"].apply(ast.literal_eval)

    rng = np.random.default_rng(seed)

    if stratify_by_source:
        train_ids, val_ids = [], []
        for _, grp in df.groupby("source"):
            ids = list(grp["image_id"].unique())
            perm = rng.permutation(len(ids))
            ids = [ids[i] for i in perm]
            n_val = max(1, int(len(ids) * val_fraction))
            val_ids.extend(ids[:n_val])
            train_ids.extend(ids[n_val:])
    else:
        ids = list(df["image_id"].unique())
        perm = rng.permutation(len(ids))
        ids = [ids[i] for i in perm]
        n_val = int(len(ids) * val_fraction)
        val_ids, train_ids = ids[:n_val], ids[n_val:]

    assert len(set(train_ids) & set(val_ids)) == 0, "train/valid image_id overlap!"
    assert len(train_ids) + len(val_ids) == df["image_id"].nunique()

    train_coco = build_split(df, train_ids)
    val_coco = build_split(df, val_ids)

    with open(os.path.join(out_dir, "train.json"), "w") as f:
        json.dump(train_coco, f)
    with open(os.path.join(out_dir, "valid.json"), "w") as f:
        json.dump(val_coco, f)

    print(f"Train: {len(train_coco['images'])} images, {len(train_coco['annotations'])} boxes")
    print(f"Valid: {len(val_coco['images'])} images, {len(val_coco['annotations'])} boxes")
    print(f"Written to {out_dir}/train.json and {out_dir}/valid.json")

In [ ]:
convert(
    csv_path="/kaggle/input/competitions/global-wheat-detection/train.csv",
    out_dir="/kaggle/working/outputs/coco_dataset",
    val_fraction=0.1,
    seed=42,
)

## Dino DETR architecture

In [ ]:
# ============================================================
# Deformable DETR baseline (DINO's direct architectural ancestor)
# via Hugging Face transformers — no compiled CUDA kernels needed.
# Reuses train.json / valid.json already built by csv_to_coco.py
# ============================================================

!pip install -U transformers torchmetrics pycocotools
!nvcc --version

import os
import torch
from PIL import Image
from pycocotools.coco import COCO
from transformers import (
    AutoImageProcessor,
    DeformableDetrForObjectDetection,
    TrainingArguments,
    Trainer,
)

CHECKPOINT = "SenseTime/deformable-detr"
IMG_ROOT = "/kaggle/input/competitions/global-wheat-detection/train/"
ANN_ROOT = "/kaggle/working/outputs/coco_dataset/"


def build_dataset(ann_file: str, img_root: str):
    """Wrap a COCO json + image folder into a torch Dataset of RAW (unprocessed) items."""
    coco = COCO(ann_file)
    img_ids = list(coco.imgs.keys())

    class CocoDetectionDataset(torch.utils.data.Dataset):
        def __len__(self):
            return len(img_ids)

        def __getitem__(self, idx):
            img_id = img_ids[idx]
            img_info = coco.imgs[img_id]
            image = Image.open(os.path.join(img_root, img_info["file_name"])).convert("RGB")

            ann_ids = coco.getAnnIds(imgIds=img_id)
            anns = coco.loadAnns(ann_ids)
            target = {"image_id": img_id, "annotations": anns}

            return {"image": image, "target": target}  # raw, not yet processed

    return CocoDetectionDataset()


def collate_fn(batch, processor):
    """Let the processor batch-pad internally, instead of calling its
    (version-fragile) low-level pad helper ourselves."""
    images = [item["image"] for item in batch]
    targets = [item["target"] for item in batch]

    encoding = processor(images=images, annotations=targets, return_tensors="pt")

    return {
        "pixel_values": encoding["pixel_values"],
        "pixel_mask": encoding.get("pixel_mask"),
        "labels": encoding["labels"],
    }


def train_deformable_detr(epochs: int = 24, batch_size: int = 4, lr: float = 1e-4, fp16: bool = True):
    processor = AutoImageProcessor.from_pretrained(CHECKPOINT)
    processor.size = {"shortest_edge": 512, "longest_edge": 800} 
    train_ds = build_dataset(ANN_ROOT + "train.json", IMG_ROOT)
    val_ds = build_dataset(ANN_ROOT + "valid.json", IMG_ROOT)

    model = DeformableDetrForObjectDetection.from_pretrained(
        CHECKPOINT,
        num_labels=1,                      # wheat_head only
        ignore_mismatched_sizes=True,       # replacing the COCO 91-class head
    )
    model = model.to("cuda:0" if torch.cuda.is_available() else "cpu")

    args = TrainingArguments(
        output_dir="/kaggle/working/global_wheat/baseline_deformable_detr",
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        learning_rate=lr,
        weight_decay=1e-4,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=50,
        remove_unused_columns=False,        # required for object detection
        fp16=fp16 and torch.cuda.is_available(),
        dataloader_num_workers=4,        # or os.cpu_count() // 2
        dataloader_persistent_workers=True,  # avoid respawning workers every epoch
        dataloader_prefetch_factor=2,
    )
    # Force single-GPU: overriding this directly stops Trainer from ever
    # wrapping the model in nn.DataParallel, which is what triggers the
    # `self.device` StopIteration bug — no kernel restart needed.
    args._n_gpu = 1

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=lambda batch: collate_fn(batch, processor),
    )

    trainer.train()
    return trainer


trainer = train_deformable_detr(epochs=30, batch_size=4)

In [ ]:
import os, json
import torch
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

device = "cuda" if torch.cuda.is_available() else "cpu"
model = trainer.model.to(device).eval()

coco_gt = COCO(ANN_ROOT + "valid.json")
img_ids = list(coco_gt.imgs.keys())

processor = AutoImageProcessor.from_pretrained(CHECKPOINT)
processor.size = {"shortest_edge": 512, "longest_edge": 800}  # match whatever you trained with

results = []
with torch.no_grad():
    for img_id in img_ids:
        img_info = coco_gt.imgs[img_id]
        image = Image.open(os.path.join(IMG_ROOT, img_info["file_name"])).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(device)
        outputs = model(**inputs)

        target_sizes = torch.tensor([image.size[::-1]])  # (h, w)
        processed = processor.post_process_object_detection(
            outputs, target_sizes=target_sizes, threshold=0.0  # keep everything; COCOeval sweeps thresholds itself
        )[0]

        for box, score, label in zip(
            processed["boxes"].cpu().numpy(),
            processed["scores"].cpu().numpy(),
            processed["labels"].cpu().numpy(),
        ):
            x1, y1, x2, y2 = box
            results.append({
                "image_id": img_id,
                "category_id": int(label),
                "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                "score": float(score),
            })

with open("val_predictions.json", "w") as f:
    json.dump(results, f)

In [ ]:
coco_dt = coco_gt.loadRes("val_predictions.json")
coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()   # prints all 12 standard COCO metrics

mAP50_95 = coco_eval.stats[0]  # AP @ IoU=0.50:0.95
mAP50    = coco_eval.stats[1]  # AP @ IoU=0.50
print(f"mAP50:    {mAP50:.4f}")
print(f"mAP50-95: {mAP50_95:.4f}")

In [ ]:
from collections import defaultdict

def precision_recall_at_threshold(coco_gt, results, score_thresh=0.5, iou_thresh=0.5):
    gt_by_img = defaultdict(list)
    for ann in coco_gt.anns.values():
        x, y, w, h = ann["bbox"]
        gt_by_img[ann["image_id"]].append([x, y, x + w, y + h])

    pred_by_img = defaultdict(list)
    for r in results:
        if r["score"] < score_thresh:
            continue
        x, y, w, h = r["bbox"]
        pred_by_img[r["image_id"]].append(([x, y, x + w, y + h], r["score"]))

    def iou(a, b):
        ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
        ix1, iy1 = max(ax1, bx1), max(ay1, by1)
        ix2, iy2 = min(ax2, bx2), min(ay2, by2)
        inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
        union = (ax2-ax1)*(ay2-ay1) + (bx2-bx1)*(by2-by1) - inter
        return inter / union if union > 0 else 0

    tp = fp = fn = 0
    for img_id in coco_gt.imgs:
        gts = gt_by_img.get(img_id, []).copy()
        preds = sorted(pred_by_img.get(img_id, []), key=lambda x: -x[1])
        matched = [False] * len(gts)
        for box, _ in preds:
            best_iou, best_j = 0, -1
            for j, gt in enumerate(gts):
                if matched[j]:
                    continue
                i = iou(box, gt)
                if i > best_iou:
                    best_iou, best_j = i, j
            if best_iou >= iou_thresh:
                tp += 1
                matched[best_j] = True
            else:
                fp += 1
        fn += matched.count(False)

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    return precision, recall

precision, recall = precision_recall_at_threshold(coco_gt, results, score_thresh=0.5, iou_thresh=0.5)
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def compare_gt_vs_dino(image_id=None, score_thresh=0.5, coco_gt=coco_gt, img_root=IMG_ROOT,
                        model=model, processor=processor, device=device):
    if image_id is None:
        image_id = random.choice(list(coco_gt.imgs.keys()))

    img_info = coco_gt.imgs[image_id]
    image = Image.open(os.path.join(img_root, img_info["file_name"])).convert("RGB")

    # ---- Ground-truth boxes (COCO format: [x, y, w, h]) ----
    ann_ids = coco_gt.getAnnIds(imgIds=image_id)
    gt_anns = coco_gt.loadAnns(ann_ids)

    # ---- DINO / Deformable DETR predictions ----
    model.eval()
    with torch.no_grad():
        inputs = processor(images=image, return_tensors="pt").to(device)
        outputs = model(**inputs)
        target_sizes = torch.tensor([image.size[::-1]])  # (h, w)
        processed = processor.post_process_object_detection(
            outputs, target_sizes=target_sizes, threshold=score_thresh
        )[0]

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    # ---- Left: ground truth ----
    axes[0].imshow(image)
    for ann in gt_anns:
        x, y, w, h = ann["bbox"]
        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor="lime", facecolor="none")
        axes[0].add_patch(rect)
    axes[0].set_title(f"Ground truth | {img_info['file_name']}\n{len(gt_anns)} boxes")
    axes[0].axis("off")

    # ---- Right: model predictions ----
    axes[1].imshow(image)
    boxes = processed["boxes"].cpu().numpy()
    scores = processed["scores"].cpu().numpy()
    for (x1, y1, x2, y2), score in zip(boxes, scores):
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="red", facecolor="none")
        axes[1].add_patch(rect)
        axes[1].text(x1, y1 - 3, f"{score:.2f}", color="red", fontsize=8,
                     bbox=dict(facecolor="white", alpha=0.6, pad=0, edgecolor="none"))
    axes[1].set_title(f"Deformable DETR predictions (score > {score_thresh})\n{len(boxes)} boxes")
    axes[1].axis("off")

    plt.suptitle("Ground truth vs. Deformable DETR predictions")
    plt.tight_layout()
    plt.show()

    return image_id

# Random validation image
compare_gt_vs_dino()

# Or a specific one, e.g. to inspect a particular image:
# compare_gt_vs_dino(image_id=some_image_id)

In [ ]:
# Ensure you have the latest version supporting YOLO26
!pip install -U ultralytics

import os
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Define models to compare (Updated to YOLO26)
models_to_compare = {
    "YOLO26n": "yolo26n.pt",   # Fastest, smallest
    "YOLO26s": "yolo26s.pt",   # Balanced baseline
    "YOLO26m": "yolo26m.pt",   # More accurate, larger
}

results = {}

# 2. Training Loop
for name, model_path in models_to_compare.items():
    print(f"\n{'='*60}")
    print(f"Training {name}...")
    print(f"{'='*60}\n")
    
    model = YOLO(model_path)
    
    results[name] = model.train(
        data="/kaggle/working/outputs/yolo_dataset/data.yaml",
        epochs=30,
        imgsz=640,
        batch=16,
        project="global_wheat_week2",
        name=f"compare_{name.lower()}",
        verbose=False
    )

In [ ]:
# 3. Validation Loop
validation_metrics = {}

for name in models_to_compare.keys():
    print(f"\nEvaluating {name}...")
    
    # Load the trained model
    model = YOLO(f"runs/detect/global_wheat_week2/compare_{name.lower()}/weights/best.pt")
    
    # Validate
    metrics = model.val(
        data="/kaggle/working/outputs/yolo_dataset/data.yaml",
        imgsz=640,
        verbose=False
    )
    
    validation_metrics[name] = {
        "mAP@50": metrics.box.map50,
        "mAP@50-95": metrics.box.map,
        "Precision": metrics.box.mp,
        "Recall": metrics.box.mr,
    }
    
    print(f"  mAP@50:    {metrics.box.map50:.4f}")
    print(f"  mAP@50-95: {metrics.box.map:.4f}")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall:    {metrics.box.mr:.4f}")

# 4. Create comparison table
comparison_df = pd.DataFrame(validation_metrics).T
print("\n" + "="*70)
print("           MODEL COMPARISON (Validation Set)")
print("="*70)
display(comparison_df)

In [ ]:
# 5. Create comparison plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics_to_plot = ["mAP@50", "mAP@50-95", "Precision", "Recall"]
colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#FFA07A"]

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]
    
    # Dynamically pulls values for YOLO26n, YOLO26s, YOLO26m
    values = [validation_metrics[model][metric] for model in validation_metrics.keys()]
    
    bars = ax.bar(validation_metrics.keys(), values, color=colors[idx], width=0.6)
    ax.set_ylabel(metric, fontweight='bold')
    ax.set_title(f"{metric} Comparison", fontweight='bold')
    ax.set_ylim(0, 1.05) # Slightly higher ceiling to make room for text labels
    
    # Add value labels on top of bars
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width()/2, 
            bar.get_height() + 0.02, 
            f'{val:.3f}', 
            ha='center', 
            va='bottom', 
            fontsize=10,
            fontweight='bold'
        )
    
    # Ensure x-axis labels are clean and readable
    ax.tick_params(axis='x', labelsize=11)

plt.suptitle("YOLO26 Model Comparison on Validation Set", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
for name in models_to_compare.keys():
    model_dir = f"runs/detect/global_wheat_week2/compare_{name.lower()}"
    best_model = f"{model_dir}/weights/best.pt"
    
    if os.path.exists(best_model):
        size_mb = os.path.getsize(best_model) / (1024 * 1024)
        
        try:
            results_csv = f"{model_dir}/results.csv"
            if os.path.exists(results_csv):
                df = pd.read_csv(results_csv)
                epochs_trained = len(df)
                
                if 'time' in df.columns:
                    # FIX: Get the LAST value (total cumulative time), not sum
                    total_time_seconds = df['time'].iloc[-1]  # Last row
                    total_time_minutes = total_time_seconds / 60
                    time_str = f"{total_time_minutes:.2f} min"
                    
                    print(f"{name:10s} | Size: {size_mb:6.2f} MB | Epochs: {epochs_trained} | Time: {time_str}")
                else:
                    print(f"{name:10s} | Time column not found")
            else:
                print(f"{name:10s} | Results.csv not found")
        except Exception as e:
            print(f"{name:10s} | Error: {e}")
    else:
        print(f"{name:10s} | Model not found")

## Summary & Next Steps

- Area-ratio and aspect-ratio outlier thresholds were calibrated visually (Section 7.2), not guessed
- Duplicate/interchangeable boxes (same wheat head annotated twice) detected via pairwise IoU per image; one copy of each pair is now excluded from training labels (Section 9.1)
- Cleaned, attribute-separated box data saved to `outputs/train_boxes_clean.csv`, including `is_outlier`, `is_duplicate`, and `use_for_training` flags
- A source-stratified, image-level train/val split (85/15) was built and saved to `outputs/image_split.csv`, so the same image never appears on both sides
- An Albumentations augmentation pipeline (`train_transform`) was built and visually verified against real bounding boxes across images with different box densities — it stays a reusable pipeline object, applied on-the-fly to the train split only, not baked into static files
- YOLO-format dataset (`images/{train,val}` + `labels/{train,val}` + `data.yaml`) written to `outputs/yolo_dataset/`, excluding area/negative-dim outliers and dropped duplicate boxes

**Week 3 & 4** will pick up from `outputs/yolo_dataset/` and `train_transform` to
train a baseline detector (e.g. YOLOv5/v8 or Faster R-CNN) on the train split,
evaluate honestly on the untouched val split, and iterate on fine-tuning and
optimization (anchor tuning, threshold sweeps, TTA, architecture comparison, etc.).

# YOLO26s Runs and Trial using augmentation, Finding Best HyperParameters

In [ ]:
!pip install -U ultralytics

from ultralytics import YOLO

# Load the latest YOLO26 small model
model = YOLO("yolo26s.pt")

results = model.train(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="global_wheat",
    name="yolo26s_augmented_fast",

    # 📉 EARLY STOPPING & LOGGING (Saves hours of wasted compute)
    patience=10,        # Stop training if validation metrics don't improve for 10 epochs
    save_period=5,      # Save checkpoints every 5 epochs (reduces disk write overhead)
    plots=True,         # Generate training plots
    exist_ok=True,      # Prevents errors if you need to re-run the same cell
)

In [ ]:
from ultralytics import YOLO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ✅ CORRECT PATH FOUND BY SEARCH
model = YOLO("/kaggle/working/runs/detect/global_wheat/yolo26s_augmented_fast/weights/best.pt")

print("="*70)
print("🎯 COMPREHENSIVE PREDICTION METRICS")
print("="*70)

# Run predictions on validation set
results = model.predict(
    source="/kaggle/working/outputs/yolo_dataset/images/val", # <--- THE FIX
    imgsz=640,
    batch=16,
    save=False,
    verbose=False,
    conf=0.25 # Only count predictions with >25% confidence
)

# Collect prediction data
all_confidences = []
all_boxes = []
total_predictions = 0
images_with_predictions = 0
images_without_predictions = 0

for result in results:
    total_predictions += len(result.boxes) if result.boxes is not None else 0
    if result.boxes is not None and len(result.boxes) > 0:
        images_with_predictions += 1
        all_confidences.extend(result.boxes.conf.cpu().numpy().tolist())
        all_boxes.extend(result.boxes.xyxy.cpu().numpy().tolist())
    else:
        images_without_predictions += 1

print(f"\n📊 PREDICTION STATISTICS")
print(f"{'='*70}")
print(f"Total images processed: {len(results)}")
print(f"Images with predictions: {images_with_predictions}")
print(f"Images without predictions: {images_without_predictions}")
print(f"Total bounding boxes predicted: {total_predictions}")
print(f"Average boxes per image: {total_predictions/len(results):.2f}")

if all_confidences:
    print(f"\n📈 CONFIDENCE SCORE ANALYSIS")
    print(f"{'='*70}")
    print(f"Mean confidence: {np.mean(all_confidences):.4f}")
    print(f"Median confidence: {np.median(all_confidences):.4f}")
    print(f"Min confidence: {np.min(all_confidences):.4f}")
    print(f"Max confidence: {np.max(all_confidences):.4f}")
    
    high_conf = sum(1 for c in all_confidences if c >= 0.75)
    med_conf = sum(1 for c in all_confidences if 0.5 <= c < 0.75)
    low_conf = sum(1 for c in all_confidences if c < 0.5)
    
    print(f"\nConfidence Distribution:")
    print(f"  High (≥0.75): {high_conf} ({high_conf/len(all_confidences)*100:.1f}%)")
    print(f"  Medium (0.5-0.75): {med_conf} ({med_conf/len(all_confidences)*100:.1f}%)")
    print(f"  Low (<0.5): {low_conf} ({low_conf/len(all_confidences)*100:.1f}%)")

# Box size analysis
if all_boxes:
    print(f"\n📦 BOUNDING BOX SIZE ANALYSIS")
    print(f"{'='*70}")
    
    box_sizes = []
    box_aspect_ratios = []
    
    for box in all_boxes:
        x1, y1, x2, y2 = box
        width = x2 - x1
        height = y2 - y1
        area = width * height
        aspect_ratio = width / (height + 1e-6)
        
        box_sizes.append(area)
        box_aspect_ratios.append(aspect_ratio)
    
    print(f"Mean box area: {np.mean(box_sizes):.2f} pixels²")
    print(f"Median box area: {np.median(box_sizes):.2f} pixels²")
    print(f"Min box area: {np.min(box_sizes):.2f} pixels²")
    print(f"Max box area: {np.max(box_sizes):.2f} pixels²")
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.hist(box_sizes, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    plt.xlabel('Box Area (pixels²)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Bounding Box Sizes')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.hist(box_aspect_ratios, bins=50, edgecolor='black', alpha=0.7, color='coral')
    plt.xlabel('Aspect Ratio (width/height)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Box Aspect Ratios')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Confidence vs Box size correlation
if all_confidences and all_boxes:
    print(f"\n🔍 CONFIDENCE vs BOX SIZE CORRELATION")
    print(f"{'='*70}")
    
    box_areas = [(b[2] - b[0]) * (b[3] - b[1]) for b in all_boxes]
    correlation = np.corrcoef(box_areas, all_confidences)[0, 1]
    print(f"Correlation between box size and confidence: {correlation:.4f}")
    
    if correlation > 0.3:
        print("⚠️  Model is more confident on LARGER objects (might miss tiny wheat heads)")
    elif correlation < -0.3:
        print("⚠️  Model is more confident on SMALLER objects")
    else:
        print("✅ Confidence is independent of object size (Excellent!)")
    
    plt.figure(figsize=(10, 6))
    plt.scatter(box_areas, all_confidences, alpha=0.5, s=20, color='purple')
    plt.xlabel('Box Area (pixels²)')
    plt.ylabel('Confidence Score')
    plt.title('Confidence vs Object Size')
    plt.grid(True, alpha=0.3)
    plt.show()

# Final Validation Metrics
print(f"\n📊 FINAL VALIDATION METRICS")
print(f"{'='*70}")
val_metrics = model.val(data="/kaggle/working/outputs/yolo_dataset/data.yaml", imgsz=640, verbose=False)

print(f"mAP@50:    {val_metrics.box.map50:.4f}")
print(f"mAP@75:    {val_metrics.box.map75:.4f}")
print(f"mAP@50-95: {val_metrics.box.map:.4f}")
print(f"Precision: {val_metrics.box.mp:.4f}")
print(f"Recall:    {val_metrics.box.mr:.4f}")
f1 = 2 * (val_metrics.box.mp * val_metrics.box.mr) / (val_metrics.box.mp + val_metrics.box.mr + 1e-16)
print(f"F1-Score:  {f1:.4f}")
print(f"{'='*70}")

In [ ]:
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
import random

print("="*70)
print("🖼️ VISUALIZING SAMPLE PREDICTIONS")
print("="*70)

# ✅ CORRECT PATH: images/val (not valid/images)
val_dir = Path("/kaggle/working/outputs/yolo_dataset/images/val")
print(f" Checking directory: {val_dir}")
print(f"✅ Directory exists: {val_dir.exists()}")

# Get images (Check for BOTH .jpg and .png)
val_images = list(val_dir.glob("*.jpg")) + list(val_dir.glob("*.png"))
print(f"🖼️ Found {len(val_images)} validation images in total.\n")

if len(val_images) == 0:
    print("❌ ERROR: No images found!")
    print("💡 Check if the path is correct")
else:
    # Pick up to 8 random images
    num_to_show = min(8, len(val_images))
    sample_images = random.sample(val_images, num_to_show)

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()

    for idx, img_path in enumerate(sample_images):
        # Run prediction
        result = model.predict(str(img_path), imgsz=640, conf=0.25, verbose=False)[0]
        
        # Get image with boxes drawn
        plotted_img = result.plot()
        
        # Convert BGR (OpenCV) to RGB (Matplotlib)
        img_rgb = cv2.cvtColor(plotted_img, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(img_rgb)
        axes[idx].axis('off')
        
        # Count boxes and add to title
        num_boxes = len(result.boxes) if result.boxes is not None else 0
        short_name = img_path.name[:15] + "..." if len(img_path.name) > 15 else img_path.name
        axes[idx].set_title(f"{short_name}\nPredictions: {num_boxes}", fontsize=10, fontweight='bold')

    # Hide unused subplots
    for idx in range(num_to_show, 8):
        axes[idx].axis('off')

    plt.tight_layout()
    plt.show()
    print("\n✅ Visualization complete!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("="*70)
print("📈 TRAINING HEALTH & NEXT STEPS")
print("="*70)

# ✅ CORRECT PATH FOUND BY SEARCH
results_csv = Path("/kaggle/working/runs/detect/global_wheat/yolo26s_augmented_fast/results.csv")

if results_csv.exists():
    df = pd.read_csv(results_csv)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot mAP
    axes[0].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@50-95', color='blue', linewidth=2)
    axes[0].set_title('Model Accuracy Over Time')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('mAP@50-95')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot Loss
    axes[1].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss', color='orange')
    axes[1].plot(df['epoch'], df['val/box_loss'], label='Val Box Loss', color='red', linewidth=2)
    axes[1].set_title('Loss Over Time')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Actionable Advice
    print("\n💡 ANALYSIS & NEXT STEPS:")
    print("-" * 70)
    
    train_loss_end = df['train/box_loss'].iloc[-1]
    val_loss_end = df['val/box_loss'].iloc[-1]
    
    if val_loss_end > train_loss_end * 1.3:
        print("⚠️ OVERFITTING DETECTED: Validation loss is much higher than training loss.")
        print("   ➔ FIX: Add slight dropout, reduce epochs, or add very light augmentation.")
    else:
        print("✅ HEALTHY TRAINING: Train and Validation losses are well-aligned.")
        
    last_5_epochs_map = df['metrics/mAP50-95(B)'].iloc[-5:].mean()
    if df['metrics/mAP50-95(B)'].iloc[-1] > last_5_epochs_map + 0.005:
        print("📈 TREND: mAP is still rising at the end of training.")
        print("   ➔ FIX: You could likely get better results by training for more epochs (e.g., 100).")
    else:
        print("✅ CONVERGENCE: Model has fully converged. No need for more epochs.")

else:
    print("❌ Could not find results.csv. Please check the path.")

In [ ]:
model = YOLO("yolo26s.pt")

results = model.train(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="global_wheat",
    name="yolo26s_tuned_week4_boxfix",
    exist_ok=True,

    box=8.5,            # FIXED — was 0.09, now correctly ABOVE default (7.5) to push tighter boxes
    dfl=1.75,            # slightly softened from your 2.0, paired properly with box now
    cls=0.5,             # default, unchanged
    lr0=0.01,
    lrf=0.005,
    weight_decay=0.0005,

    patience=10,
    save_period=5,
    plots=True,
)

In [ ]:
from ultralytics import YOLO

# Load your tuned model
model = YOLO("/kaggle/working/runs/detect/global_wheat/yolo26s_tuned_week4_boxfix/weights/best.pt")

print(" Running Test-Time Augmentation (TTA) Ensemble...")
print("="*70)

# Standard validation
standard = model.val(data="/kaggle/working/outputs/yolo_dataset/data.yaml", 
                     imgsz=640, verbose=False)

# TTA validation (looks at flipped/scaled versions too)
tta = model.val(data="/kaggle/working/outputs/yolo_dataset/data.yaml", 
                imgsz=640, augment=True, verbose=False)

print("\n📊 COMPARISON: Standard vs TTA Ensemble")
print("="*70)
print(f"{'Metric':<15} {'Standard':<15} {'TTA':<15} {'Improvement':<15}")
print("-"*70)

std_mAP50 = standard.box.map50
std_mAP75 = standard.box.map75
std_mAP = standard.box.map
std_f1 = 2 * (standard.box.mp * standard.box.mr) / (standard.box.mp + standard.box.mr + 1e-16)

tta_mAP50 = tta.box.map50
tta_mAP75 = tta.box.map75
tta_mAP = tta.box.map
tta_f1 = 2 * (tta.box.mp * tta.box.mr) / (tta.box.mp + tta.box.mr + 1e-16)

print(f"{'mAP@50':<15} {std_mAP50:<15.4f} {tta_mAP50:<15.4f} {tta_mAP50-std_mAP50:<15.4f}")
print(f"{'mAP@75':<15} {std_mAP75:<15.4f} {tta_mAP75:<15.4f} {tta_mAP75-std_mAP75:<15.4f}")
print(f"{'mAP@50-95':<15} {std_mAP:<15.4f} {tta_mAP:<15.4f} {tta_mAP-std_mAP:<15.4f}")
print(f"{'F1-Score':<15} {std_f1:<15.4f} {tta_f1:<15.4f} {tta_f1-std_f1:<15.4f}")
print("="*70)

In [ ]:
from ultralytics import YOLO

# Load the model
model = YOLO("yolo26s.pt")

# Run the automated hyperparameter search
model.tune(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    epochs=10,         # Keep low (10) just to find the best settings quickly
    iterations=10,     # Tests 10 different combinations
    imgsz=480,         # Keep low (480) to make the search run much faster
    batch=32,          # High batch for speed
    optimizer="AdamW", # Good optimizer for tuning
    plots=False,       # Save time by not generating plots during search
    save=False,        # Don't save the 10 temporary models, just the results
    project="global_wheat",
    name="hyperparameter_search",
    exist_ok=True,
)

In [ ]:
import yaml
from pathlib import Path

print("="*70)
print("🏆 EXTRACTING BEST HYPERPARAMETERS")
print("="*70)

# Path to the best hyperparameters YAML file
yaml_path = "/kaggle/working/runs/detect/global_wheat/hyperparameter_search/best_hyperparameters.yaml"

if Path(yaml_path).exists():
    with open(yaml_path, 'r') as f:
        best_params = yaml.safe_load(f)
    
    print("✅ Best hyperparameters loaded!")
    print(f"Best fitness (mAP@50-95): {best_params.get('fitness', 'N/A')}")
    print("\n📋 Copy these parameters for your final training:")
    print("-" * 70)
    
    # Print the key parameters
    key_params = ['lr0', 'lrf', 'momentum', 'weight_decay', 'box', 'cls', 'dfl', 
                  'hsv_h', 'hsv_s', 'hsv_v', 'degrees', 'translate', 'scale', 
                  'shear', 'perspective', 'flipud', 'fliplr', 'mosaic', 'mixup', 'copy_paste']
    
    for param in key_params:
        if param in best_params:
            print(f"    {param}={best_params[param]},")
    
    print("-" * 70)
    print("\n Use these in your final model.train() call!")
else:
    print(f"❌ YAML file not found at: {yaml_path}")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")

best_hyp = {
    "lr0": 0.01,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.00061,
    "box": 11.13557,
    "cls": 1.26513,
    "dfl": 1.5,
    "hsv_h": 0.015,
    "hsv_s": 0.82952,
    "hsv_v": 0.37353,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.31877,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 0.90701,
    "mixup": 0.0,
    "copy_paste": 0.0,
}

results = model.train(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    **best_hyp,
)

# Best Hyperparamter (Round 1) When epochs = 10 and iter = 30 and resolution = 768px

In [ ]:
from ultralytics import YOLO

# Load the model
model = YOLO("yolo26s.pt")

# Run the automated hyperparameter search
model.tune(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    epochs=10,         # Keep low (10) just to find the best settings quickly
    iterations=30,     # Tests 30 different combinations
    imgsz=768,         # Keep low (480) to make the search run much faster
    batch=32,          # High batch for speed
    optimizer="AdamW", # Good optimizer for tuning
    plots=False,       # Save time by not generating plots during search
    save=False,        # Don't save the 10 temporary models, just the results
    project="global_wheat",
    name="hyperparameter_search",
    exist_ok=True,
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")

best_hyp = {
    "lr0": 0.00619, "lrf": 0.11908, "momentum": 0.94055, "weight_decay": 0.0007,
    "warmup_epochs": 3.47727, "warmup_momentum": 0.84631,
    "box": 1.0, "cls": 0.1, "cls_pw": 0.12554, "dfl": 0.4,
    "hsv_h": 0.01005, "hsv_s": 0.66883, "hsv_v": 0.9,
    "degrees": 4.35793, "translate": 0.1706, "scale": 0.20061, "shear": 1.23097,
    "perspective": 0.0, "flipud": 0.0, "fliplr": 0.20395, "bgr": 0.09931,
    "mosaic": 0.83444, "mixup": 0.10809, "cutmix": 0.0, "copy_paste": 0.0,
    "close_mosaic": 6,
}

results = model.train(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    **best_hyp,
)

## Correction [box = 8.5]

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")

best_hyp = {
    "lr0": 0.00619, "lrf": 0.11908, "momentum": 0.94055, "weight_decay": 0.0007,
    "warmup_epochs": 3.47727, "warmup_momentum": 0.84631,
    "box": 7.5, "cls": 0.1, "cls_pw": 0.12554, "dfl": 0.4,
    "hsv_h": 0.01005, "hsv_s": 0.66883, "hsv_v": 0.9,
    "degrees": 4.35793, "translate": 0.1706, "scale": 0.20061, "shear": 1.23097,
    "perspective": 0.0, "flipud": 0.0, "fliplr": 0.20395, "bgr": 0.09931,
    "mosaic": 0.83444, "mixup": 0.10809, "cutmix": 0.0, "copy_paste": 0.0,
    "close_mosaic": 6,
}

results = model.train(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    **best_hyp,
)

# Best Hyperparamter (Round 2) When epochs = 20 , iter = 30 and resolution = 640px 

In [ ]:
from ultralytics import YOLO

# Load the model
model = YOLO("yolo26s.pt")

# Run the automated hyperparameter search
model.tune(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    epochs=10,         # Keep low (10) just to find the best settings quickly
    iterations=30,     # Tests 30 different combinations
    imgsz=768,         # Keep low (480) to make the search run much faster
    batch=32,          # High batch for speed
    optimizer="AdamW", # Good optimizer for tuning
    plots=False,       # Save time by not generating plots during search
    save=False,        # Don't save the 10 temporary models, just the results
    project="global_wheat",
    name="hyperparameter_search",
    exist_ok=True,
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")

best_hyp = {
    "lr0": 0.01, "lrf": 0.24366, "momentum": 0.94937, "weight_decay": 0.00054,
    "warmup_epochs": 4.02764, "warmup_momentum": 0.95,
    "box": 6.80824, "cls": 0.5, "cls_pw": 0.0, "dfl": 1.5,
    "hsv_h": 0.015, "hsv_s": 0.9, "hsv_v": 0.48878,
    "degrees": 0.0, "translate": 0.06989, "scale": 0.5, "shear": 0.0,
    "perspective": 0.0, "flipud": 0.1104, "fliplr": 0.5, "bgr": 0.05886,
    "mosaic": 0.96396, "mixup": 0.0, "cutmix": 0.32769, "copy_paste": 0.0,
    "close_mosaic": 9,
}

results = model.train(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    **best_hyp,
)

# Augmenting the epochs from 50 to 100 and resolution to 1024

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO

# Load the YOLO26s model
model = YOLO("yolo26s.pt")

# Train the model using default hyperparameters
results = model.train(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    epochs=100,          # Full training run
    imgsz=1024,          # Full resolution for small wheat heads
    batch=4,             # Lowered to prevent Out-Of-Memory crash at 1024px
    project="global_wheat",
    name="train_final",
    exist_ok=True,
    plots=True,          # Generates and saves training curves/plots
    
    # --- Smart Training Settings for 100 Epochs ---
    patience=30,         # Stops training early if mAP doesn't improve for 30 epochs
    cos_lr=True,         # Uses Cosine Learning Rate decay
    close_mosaic=10,     # Disables Mosaic augmentation in the last 10 epochs
    optimizer='AdamW',   # Optimizer choice
)

In [ ]:
from ultralytics import YOLO

# Load the best model weights from the final training run
model_path = "/kaggle/working/runs/detect/global_wheat/train_final/weights/best.pt"
model = YOLO(model_path)

# Run validation specifically on the 'val' split
metrics = model.val(data="/kaggle/working/outputs/yolo_dataset/data.yaml", split="val")

# Extract the core metrics from the Ultralytics results object
precision = metrics.box.mp      # Mean Precision
recall = metrics.box.mr         # Mean Recall
mAP_50 = metrics.box.map50      # mAP @ 0.50 IoU
mAP_75 = metrics.box.map75      # mAP @ 0.75 IoU
mAP_50_95 = metrics.box.map     # mAP @ 0.50:0.95 IoU

# Calculate F1 Score (Harmonic mean of Precision and Recall)
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

# Display the results cleanly
print("-" * 30)
print("   Validation Set Metrics   ")
print("-" * 30)
print(f"Precision:  {precision:.4f}")
print(f"Recall:     {recall:.4f}")
print(f"F1 Score:   {f1_score:.4f}")
print(f"mAP@50:     {mAP_50:.4f}")
print(f"mAP@75:     {mAP_75:.4f}")
print(f"mAP@50-95:  {mAP_50_95:.4f}")
print("-" * 30)

# Learning Rates' Sweep

In [ ]:
from ultralytics import YOLO

lr_candidates = [0.00056, 0.001, 0.005, 0.00619, 0.008095, 0.01]

for lr in lr_candidates:
    model = YOLO("yolo26s.pt")
    model.train(
        data="/kaggle/working/outputs/yolo_dataset/data.yaml",
        epochs=20,
        imgsz=640,
        batch=16,
        optimizer="AdamW",   # <-- REQUIRED: pins the optimizer so lr0 is actually respected
        lr0=lr,
        momentum=0.937,      # optional but good practice to also set explicitly
        project="global_wheat",
        name=f"yolo26s_lr_{lr}",
        patience=10,
        save_period=5,
        plots=True,
        exist_ok=True,
    )

In [ ]:
!pip install ultralytics
from ultralytics import YOLO


model = YOLO("yolo26s.pt")
model.train(
        data="/kaggle/working/outputs/yolo_dataset/data.yaml",
        epochs=50,
        imgsz=1024,
        batch=16,
        optimizer="AdamW",   
        lr0=0.001, # Best learning rate found from hyperparameter search 
        momentum=0.937,      
        project="global_wheat",
        name=f"yolo26s_lr_official_run",
        patience=10,
        save_period=5,
        plots=True,
        exist_ok=True,
)

# Yolo26m Trial when resolution = 1024 and epochs = 50

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO

# 1. UPGRADE TO MEDIUM MODEL (This is the biggest accuracy boost)
model = YOLO("yolo26m.pt") 

results = model.train(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    epochs=50,          # Increased to 100
    imgsz=1024,          # Keep full resolution for small wheat heads
    batch=16,             # CRITICAL: Lowered to 4. 'm' model is much larger and needs less batch size at 1024px
    project="global_wheat",
    name="train_final_medium", # New name so it doesn't overwrite your 's' model
    exist_ok=True,
    plots=True,
)

In [ ]:
from ultralytics import YOLO

# Load the best model weights from the MEDIUM model training run
model_path = "/kaggle/working/runs/detect/global_wheat/train_final_medium/weights/best.pt"
model = YOLO(model_path)

# Run validation specifically on the 'val' split
metrics = model.val(data="/kaggle/working/outputs/yolo_dataset/data.yaml", split="val")

# Extract the core metrics from the Ultralytics results object
precision = metrics.box.mp      # Mean Precision
recall = metrics.box.mr         # Mean Recall
mAP_50 = metrics.box.map50      # mAP @ 0.50 IoU
mAP_75 = metrics.box.map75      # mAP @ 0.75 IoU
mAP_50_95 = metrics.box.map     # mAP @ 0.50:0.95 IoU

# Calculate F1 Score (Harmonic mean of Precision and Recall)
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

# Display the results cleanly
print("-" * 30)
print("   Validation Set Metrics   ")
print("-" * 30)
print(f"Precision:  {precision:.4f}")
print(f"Recall:     {recall:.4f}")
print(f"F1 Score:   {f1_score:.4f}")
print(f"mAP@50:     {mAP_50:.4f}")
print(f"mAP@75:     {mAP_75:.4f}")
print(f"mAP@50-95:  {mAP_50_95:.4f}")
print("-" * 30)

In [ ]:
from ultralytics import YOLO

# Load your best model (YOLO26m performed better)
model_path = "/kaggle/working/runs/detect/global_wheat/train_final_medium/weights/best.pt"
model = YOLO(model_path)

# Run validation on the TEST split
print("=" * 50)
print("📊 TEST SET METRICS")
print("=" * 50)

metrics = model.val(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml", 
    split="test",  # This validates on the test split
    imgsz=1024,
    batch=16,
    plots=True
)

# Extract and display metrics
precision = metrics.box.mp
recall = metrics.box.mr
mAP_50 = metrics.box.map50
mAP_75 = metrics.box.map75
mAP_50_95 = metrics.box.map
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print("\n" + "-" * 40)
print("   Test Set Performance   ")
print("-" * 40)
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1 Score:    {f1_score:.4f}")
print(f"mAP@50:      {mAP_50:.4f}")
print(f"mAP@75:      {mAP_75:.4f}")
print(f"mAP@50-95:   {mAP_50_95:.4f}  ← PRIMARY METRIC")
print("-" * 40)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path
import random
from ultralytics import YOLO

# 1. Setup paths to your internal test set (which has labels)
TEST_IMG_DIR = Path("/kaggle/working/outputs/yolo_dataset/images/test")
TEST_LBL_DIR = Path("/kaggle/working/outputs/yolo_dataset/labels/test")

# 2. Load your best trained model
model_path = "/kaggle/working/runs/detect/global_wheat/train_final_medium/weights/best.pt"
model = YOLO(model_path)

# 3. Pick 4 random images from the test set
all_test_imgs = list(TEST_IMG_DIR.glob("*.jpg"))
selected_imgs = random.sample(all_test_imgs, 4)

# Helper function to read YOLO format labels and convert to bounding boxes
def get_yolo_boxes(lbl_path, img_w, img_h):
    boxes = []
    if lbl_path.exists():
        with open(lbl_path, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) == 5:
                    cls, xc, yc, w, h = map(float, parts)
                    # Convert YOLO (center_x, center_y, w, h) to (x_min, y_min, x_max, y_max)
                    x1 = (xc - w/2) * img_w
                    y1 = (yc - h/2) * img_h
                    x2 = (xc + w/2) * img_w
                    y2 = (yc + h/2) * img_h
                    boxes.append([x1, y1, x2, y2])
    return boxes

# 4. Create the visualization grid (4 rows, 2 columns)
fig, axes = plt.subplots(4, 2, figsize=(16, 24))
fig.suptitle("Ground Truth vs Model Predictions (Test Set)", fontsize=20, fontweight='bold')

for i, img_path in enumerate(selected_imgs):
    img = Image.open(img_path)
    img_w, img_h = img.size
    img_id = img_path.stem
    
    # Get Ground Truth boxes
    lbl_path = TEST_LBL_DIR / f"{img_id}.txt"
    gt_boxes = get_yolo_boxes(lbl_path, img_w, img_h)
    
    # Get Model Predictions (using TTA for best results)
    # In the prediction loop, change:
    results = model.predict(
        source=str(img_path),
        imgsz=1024,
        augment=True,
        conf=0.25,      # Higher threshold
        iou=0.45,       # Remove overlapping boxes
        max_det=100,    # Limit max detections per image
        verbose=False
    )
    pred_boxes = results[0].boxes.xyxy.cpu().numpy()
    pred_confs = results[0].boxes.conf.cpu().numpy()
    
    # --- Plot Ground Truth (Left Column) ---
    ax_gt = axes[i, 0]
    ax_gt.imshow(img)
    ax_gt.set_title(f"Image {i+1}: Ground Truth ({len(gt_boxes)} boxes)", fontsize=14, color='green')
    for box in gt_boxes:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none')
        ax_gt.add_patch(rect)
    ax_gt.axis('off')
    
    # --- Plot Predictions (Right Column) ---
    ax_pred = axes[i, 1]
    ax_pred.imshow(img)
    ax_pred.set_title(f"Image {i+1}: Model Prediction ({len(pred_boxes)} boxes)", fontsize=14, color='red')
    for j, box in enumerate(pred_boxes):
        x1, y1, x2, y2 = box
        conf = pred_confs[j]
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none')
        ax_pred.add_patch(rect)
        # Add confidence score text
        ax_pred.text(x1, y1-5, f"{conf:.2f}", color='red', fontsize=10, weight='bold', 
                     bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))
    ax_pred.axis('off')

plt.tight_layout()
plt.show()

# YOLO26L Trial when epochs = 50 and resolution = 1024

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO

# 1. UPGRADE TO LARGE MODEL (Maximum possible accuracy)
model = YOLO("yolo26l.pt") 

results = model.train(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml",
    epochs=50,          
    imgsz=1024,          
    batch=8,             
    project="global_wheat",
    name="train_final_large", 
    exist_ok=True,
    plots=True,
    patience=15,         # Automatically stops if mAP@50-95 doesn't improve for 15 epochs
    
    # --- TWEAKS TO FOCUS ON mAP@50-95 ---
    box=8.5,             # INCREASED (Default is 7.5). Forces the model to care more about drawing tight, accurate boxes. 
                         # This directly improves mAP@75 and mAP@95.
    dfl=2.0,             # INCREASED (Default is 1.5). Distribution Focal Loss. Helps the model predict precise box boundaries.
    copy_paste=0.3,      # ADDED. This augmentation copies wheat heads and pastes them into new areas of the image. 
                         # It is a proven trick to significantly boost mAP for dense, small objects.
)

In [ ]:
from ultralytics import YOLO

# Load the best model weights from the MEDIUM model training run
model_path = "/kaggle/working/runs/detect/global_wheat/train_final_large/weights/best.pt"
model = YOLO(model_path)

# Run validation specifically on the 'val' split
metrics = model.val(data="/kaggle/working/outputs/yolo_dataset/data.yaml", split="val")

# Extract the core metrics from the Ultralytics results object
precision = metrics.box.mp      # Mean Precision
recall = metrics.box.mr         # Mean Recall
mAP_50 = metrics.box.map50      # mAP @ 0.50 IoU
mAP_75 = metrics.box.map75      # mAP @ 0.75 IoU
mAP_50_95 = metrics.box.map     # mAP @ 0.50:0.95 IoU

# Calculate F1 Score (Harmonic mean of Precision and Recall)
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

# Display the results cleanly
print("-" * 30)
print("   Validation Set Metrics   ")
print("-" * 30)
print(f"Precision:  {precision:.4f}")
print(f"Recall:     {recall:.4f}")
print(f"F1 Score:   {f1_score:.4f}")
print(f"mAP@50:     {mAP_50:.4f}")
print(f"mAP@75:     {mAP_75:.4f}")
print(f"mAP@50-95:  {mAP_50_95:.4f}")
print("-" * 30)

In [ ]:
from ultralytics import YOLO

# Load your best model (YOLO26m performed better)
model_path = "/kaggle/working/runs/detect/global_wheat/train_final_large/weights/best.pt"
model = YOLO(model_path)

# Run validation on the TEST split
print("=" * 50)
print("📊 TEST SET METRICS")
print("=" * 50)

metrics = model.val(
    data="/kaggle/working/outputs/yolo_dataset/data.yaml", 
    split="test",  # This validates on the test split
    imgsz=1024,
    batch=16,
    plots=True
)

# Extract and display metrics
precision = metrics.box.mp
recall = metrics.box.mr
mAP_50 = metrics.box.map50
mAP_75 = metrics.box.map75
mAP_50_95 = metrics.box.map
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print("\n" + "-" * 40)
print("   Test Set Performance   ")
print("-" * 40)
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1 Score:    {f1_score:.4f}")
print(f"mAP@50:      {mAP_50:.4f}")
print(f"mAP@75:      {mAP_75:.4f}")
print(f"mAP@50-95:   {mAP_50_95:.4f}  ← PRIMARY METRIC")
print("-" * 40)